In [ ]:
# ============================================================
# CENTRALIZED EXPERIMENTS: Logistic Regression, Random Forest,
# XGBoost (Seeded Runs + Random Runs)
# Dataset = Elliptic (txs_features.csv + txs_classes.csv)
# ============================================================

!pip install xgboost --quiet

import os, json, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import joblib
warnings.filterwarnings("ignore")

# ============================================================
# 1. Paths & Settings
# ============================================================

DRIVE_BASE = "/content/drive/MyDrive/PhD_FraudDetection"
os.makedirs(DRIVE_BASE, exist_ok=True)

FEATURES = "/content/txs_features.csv"
CLASSES  = "/content/txs_classes.csv"

FINAL_DATA = os.path.join(DRIVE_BASE, "elliptic_final_centralized.csv")
RESULTS_DIR = os.path.join(DRIVE_BASE, "results_elliptic_centralized")
os.makedirs(RESULTS_DIR, exist_ok=True)

SEEDS = [1011, 2022, 3033, 4044, 5055]
N_RANDOM_RUNS = 5

# ============================================================
# 2. Load & Merge Elliptic Data
# ============================================================

def load_and_merge():
    print("Loading:", FEATURES)
    df_feat = pd.read_csv(FEATURES)

    print("Loading:", CLASSES)
    df_class = pd.read_csv(CLASSES)

    if 'txId' not in df_feat.columns or 'txId' not in df_class.columns:
        raise Exception("txId missing in files.")

    df = df_feat.merge(df_class, on='txId', how='inner')
    print("Merged shape:", df.shape)

    # Class = 1 (illicit) → fraud; others → non-fraud
    df['is_fraud'] = df['class'].apply(lambda x: 1 if int(x) == 1 else 0)

    df.to_csv(FINAL_DATA, index=False)
    print("Saved merged file:", FINAL_DATA)
    return df

if not os.path.exists(FINAL_DATA):
    df = load_and_merge()
else:
    df = pd.read_csv(FINAL_DATA)
    print("Loaded merged dataset:", df.shape)

# ============================================================
# 3. Feature Selection + Scaling
# ============================================================

NUMERIC_COLS = df.select_dtypes(include=[np.number]).columns.tolist()
NUMERIC_COLS = [c for c in NUMERIC_COLS if c not in ('txId','class','is_fraud')]

print("Using numeric features:", NUMERIC_COLS)

X = df[NUMERIC_COLS].fillna(0).astype(float).values
y = df['is_fraud'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)
joblib.dump(scaler, os.path.join(RESULTS_DIR, "scaler.pkl"))

print("Total rows:", len(X), "Frauds:", y.sum(), "/", len(y))

# ============================================================
# 4. Utility Functions
# ============================================================

def compute_metrics(y_true, probs):
    labels = (probs >= 0.5).astype(int)
    tn,fp,fn,tp = confusion_matrix(y_true, labels).ravel()
    prec,rec,f1,_ = precision_recall_fscore_support(y_true, labels, average='binary', zero_division=0)

    try:
        auc = roc_auc_score(y_true, probs)
    except:
        auc = float('nan')

    acc = accuracy_score(y_true, labels)

    return {
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "precision": float(prec),
        "recall": float(rec),
        "f1": float(f1),
        "auc": float(auc),
        "accuracy": float(acc)
    }

# ============================================================
# 5. Centralized SEED-BASED Runs
# ============================================================

def run_centralized_seeded():
    summary = []

    for seed in SEEDS:
        print(f"\n==============================")
        print(f" CENTRALIZED: Seed {seed}")
        print(f"==============================")

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=seed,
            stratify=y if len(np.unique(y)) > 1 else None
        )

        # ---------------- LR ----------------
        try:
            lr = LogisticRegression(max_iter=1000, solver="liblinear")
            lr.fit(X_train, y_train)
            lr_probs = lr.predict_proba(X_test)[:,1]
        except:
            lr_probs = np.zeros(len(y_test))

        lr_m = compute_metrics(y_test, lr_probs)

        # ---------------- RF ----------------
        rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed)
        rf.fit(X_train, y_train)
        rf_probs = rf.predict_proba(X_test)[:,1]
        rf_m = compute_metrics(y_test, rf_probs)

        # ---------------- XGB ----------------
        xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=seed)
        xgb.fit(X_train, y_train)
        xgb_probs = xgb.predict_proba(X_test)[:,1]
        xgb_m = compute_metrics(y_test, xgb_probs)

        # Save JSON
        out_json = {
            "seed": seed,
            "lr": lr_m,
            "rf": rf_m,
            "xgb": xgb_m
        }

        with open(os.path.join(RESULTS_DIR, f"seed_{seed}_results.json"), "w") as f:
            json.dump(out_json, f, indent=2)

        summary.append({
            "seed": seed,
            "lr_f1": lr_m["f1"],
            "rf_f1": rf_m["f1"],
            "xgb_f1": xgb_m["f1"],
            "lr_auc": lr_m["auc"],
            "rf_auc": rf_m["auc"],
            "xgb_auc": xgb_m["auc"],
            "lr_acc": lr_m["accuracy"],
            "rf_acc": rf_m["accuracy"],
            "xgb_acc": xgb_m["accuracy"]
        })

    df_summary = pd.DataFrame(summary)
    df_summary.to_csv(os.path.join(RESULTS_DIR, "centralized_seeded_summary.csv"), index=False)

    print("\n✓ Centralized seeded summary saved.")
    return df_summary

# ============================================================
# 6. Centralized RANDOM Runs
# ============================================================

def run_centralized_random():
    rows = []

    for i in range(1, N_RANDOM_RUNS + 1):
        seed = int(np.random.randint(0, 10**6))
        print(f"\n=== RANDOM RUN {i} (seed={seed}) ===")

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=seed,
            stratify=y if len(np.unique(y)) > 1 else None
        )

        # LR
        try:
            lr = LogisticRegression(max_iter=1000, solver="liblinear")
            lr.fit(X_train, y_train)
            lr_probs = lr.predict_proba(X_test)[:,1]
        except:
            lr_probs = np.zeros(len(y_test))
        lr_m = compute_metrics(y_test, lr_probs)

        # RF
        rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed)
        rf.fit(X_train, y_train)
        rf_probs = rf.predict_proba(X_test)[:,1]
        rf_m = compute_metrics(y_test, rf_probs)

        # XGB
        xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=seed)
        xgb.fit(X_train, y_train)
        xgb_probs = xgb.predict_proba(X_test)[:,1]
        xgb_m = compute_metrics(y_test, xgb_probs)

        rows.append({
            "run": i,
            "seed": seed,
            "lr_f1": lr_m["f1"],
            "rf_f1": rf_m["f1"],
            "xgb_f1": xgb_m["f1"],
            "lr_auc": lr_m["auc"],
            "rf_auc": rf_m["auc"],
            "xgb_auc": xgb_m["auc"],
            "lr_acc": lr_m["accuracy"],
            "rf_acc": rf_m["accuracy"],
            "xgb_acc": xgb_m["accuracy"]
        })

    df_random = pd.DataFrame(rows)
    df_random.to_csv(os.path.join(RESULTS_DIR, "centralized_random_summary.csv"), index=False)

    print("\n✓ Centralized random-run summary saved.")
    return df_random

# ============================================================
# 7. RUN BOTH PARTS
# ============================================================

print("\n======= RUNNING CENTRALIZED SEED-BASED EXPERIMENTS =======")
seeded_results = run_centralized_seeded()

print("\n======= RUNNING CENTRALIZED RANDOM-RUN EXPERIMENTS =======")
random_results = run_centralized_random()

print("\nAll centralized experiments completed successfully!")
print("Check results in:", RESULTS_DIR)


Loading: /content/txs_features.csv


FileNotFoundError: [Errno 2] No such file or directory: '/content/txs_features.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

ELLIPTIC_DIR = "/content/drive/MyDrive/PhD_FraudDetection/data/defi_elliptic"
FEATURES = os.path.join(ELLIPTIC_DIR, "txs_features.csv")
CLASSES  = os.path.join(ELLIPTIC_DIR, "txs_classes.csv")
FINAL_DATA = os.path.join(ELLIPTIC_DIR, "elliptic_final_dataset.csv")

print("Checking files...")
print("FEATURES exists:", os.path.exists(FEATURES))
print("CLASSES exists:", os.path.exists(CLASSES))


ValueError: Mountpoint must not already contain files

In [ ]:
import pandas as pd
import os

ELLIPTIC_DIR = "/content/drive/MyDrive/PhD_FraudDetection/data/defi_elliptic"
FEATURES = os.path.join(ELLIPTIC_DIR, "txs_features.csv")
CLASSES  = os.path.join(ELLIPTIC_DIR, "txs_classes.csv")
FINAL_DATA = os.path.join(ELLIPTIC_DIR, "elliptic_final_dataset.csv")

print("FEATURES exists:", os.path.exists(FEATURES))
print("CLASSES exists:", os.path.exists(CLASSES))


FEATURES exists: False
CLASSES exists: False


In [ ]:
import os

search_names = ["txs_features.csv", "txs_classes.csv"]

matches = []

for root, dirs, files in os.walk("/content/drive", topdown=True):
    for f in files:
        if f in search_names:
            matches.append(os.path.join(root, f))

matches


[]

In [ ]:
import os

keywords = ["elliptic", "txs", ".xlsx", ".csv"]

found = []

for root, dirs, files in os.walk("/content/drive", topdown=True):
    for f in files:
        if any(k.lower() in f.lower() for k in keywords):
            found.append(os.path.join(root, f))

found


[]

In [ ]:
import os

keywords = ["elliptic", "txs", ".xlsx", ".csv"]

found = []

for root, dirs, files in os.walk("/content/drive", topdown=True):
    for f in files:
        if any(k.lower() in f.lower() for k in keywords):
            found.append(os.path.join(root, f))

found


[]

In [ ]:
# ============================================================
# STEP 1 — MOUNT DRIVE AND LOCATE ELLIPTIC DATASET
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/PhD_FraudDetection/data/defi"

print("Searching for Elliptic dataset inside:", BASE_DIR)

# Automatically detect files
features_file = None
classes_file  = None

for root, dirs, files in os.walk(BASE_DIR):
    for f in files:
        if "features" in f.lower() and f.lower().endswith(".csv"):
            features_file = os.path.join(root, f)
        if "classes" in f.lower() and f.lower().endswith(".csv"):
            classes_file = os.path.join(root, f)

print("Found Features file :", features_file)
print("Found Classes file  :", classes_file)

if features_file is None or classes_file is None:
    raise FileNotFoundError("❌ Elliptic dataset files not found. "
                            "Upload txs_features.csv and txs_classes.csv into the folder.")


# ============================================================
# STEP 2 — LOAD DATASETS
# ============================================================
print("\nLoading datasets...")
df_feat   = pd.read_csv(features_file)
df_class  = pd.read_csv(classes_file)

print("Features shape:", df_feat.shape)
print("Classes shape :", df_class.shape)
print("Feature cols:", df_feat.columns.tolist())
print("Class cols  :", df_class.columns.tolist())


# ============================================================
# STEP 3 — MERGE ON txId
# ============================================================
print("\nMerging on txId...")
df = df_feat.merge(df_class, on="txId", how="inner")

print("Merged shape:", df.shape)
print("Class distribution:")
print(df["class"].value_counts(dropna=False))


# ============================================================
# STEP 4 — SAVE FINAL CLEAN CSV FOR ALL FUTURE WORK
# ============================================================
FINAL_CSV = "/content/drive/MyDrive/PhD_FraudDetection/data/defi/elliptic_final.csv"
df.to_csv(FINAL_CSV, index=False)

print("\n✔ Saved final unified dataset at:")
print(FINAL_CSV)


ValueError: Mountpoint must not already contain files

In [ ]:
drive.mount('/content/drive')


ValueError: Mountpoint must not already contain files

In [ ]:
print("Drive already mounted.")


Drive already mounted.


In [ ]:
# Drive already mounted earlier
print("Drive is already mounted. Proceeding...")

import os
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/PhD_FraudDetection/data/defi"

print("Searching for Elliptic dataset inside:", BASE_DIR)

# Automatically detect files
features_file = None
classes_file  = None

for root, dirs, files in os.walk(BASE_DIR):
    for f in files:
        if "features" in f.lower() and f.lower().endswith(".csv"):
            features_file = os.path.join(root, f)
        if "classes" in f.lower() and f.lower().endswith(".csv"):
            classes_file = os.path.join(root, f)

print("Found Features file :", features_file)
print("Found Classes file  :", classes_file)

if features_file is None or classes_file is None:
    raise FileNotFoundError("❌ Elliptic dataset files not found. "
                            "Upload txs_features.csv and txs_classes.csv into the folder.")


# Load datasets
print("\nLoading datasets...")
df_feat   = pd.read_csv(features_file)
df_class  = pd.read_csv(classes_file)

print("Features shape:", df_feat.shape)
print("Classes shape :", df_class.shape)

# Merge
print("\nMerging on txId...")
df = df_feat.merge(df_class, on="txId", how="inner")

print("Merged shape:", df.shape)
print("Class distribution:")
print(df["class"].value_counts(dropna=False))

# Save final dataset
FINAL_CSV = "/content/drive/MyDrive/PhD_FraudDetection/data/defi/elliptic_final.csv"
df.to_csv(FINAL_CSV, index=False)

print("\n✔ Saved final unified dataset at:")
print(FINAL_CSV)


Drive is already mounted. Proceeding...
Searching for Elliptic dataset inside: /content/drive/MyDrive/PhD_FraudDetection/data/defi
Found Features file : None
Found Classes file  : None


FileNotFoundError: ❌ Elliptic dataset files not found. Upload txs_features.csv and txs_classes.csv into the folder.

In [ ]:
import os

search_terms = ["feature", "class", "elliptic", "txs"]
drive_root = "/content/drive/MyDrive"

print("Searching entire Drive for elliptic files...\n")

found_files = []

for root, dirs, files in os.walk(drive_root):
    for f in files:
        lower_f = f.lower()
        if any(term in lower_f for term in search_terms):
            full_path = os.path.join(root, f)
            found_files.append(full_path)
            print(full_path)

if not found_files:
    print("❌ No related files found. You may have uploaded to wrong Drive folder.")
else:
    print("\n✔ Total files found:", len(found_files))


Searching entire Drive for elliptic files...

❌ No related files found. You may have uploaded to wrong Drive folder.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

src_features = "/content/txs_features.csv"
src_classes = "/content/txs_classes.csv"

dst_folder = "/content/drive/MyDrive/PhD_FraudDetection/data/defi/"

# Copy to Drive
shutil.copy(src_features, dst_folder)
shutil.copy(src_classes, dst_folder)

print("✔ Files copied to Drive folder:", dst_folder)


ValueError: Mountpoint must not already contain files

In [ ]:
/content/drive/MyDrive/PhD_FraudDetection/data/defi


NameError: name 'content' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = "/content/drive/MyDrive/PhD_FraudDetection"
for root, dirs, files in os.walk(BASE):
    for f in files:
        if "elliptic" in f.lower() or "txs" in f.lower():
            print(os.path.join(root, f))


ValueError: Mountpoint must not already contain files

In [ ]:
!ls -lh "/content/drive/MyDrive/PhD_FraudDetection/data/defi"


ls: cannot access '/content/drive/MyDrive/PhD_FraudDetection/data/defi': No such file or directory


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!ls "/content/drive/MyDrive"


ValueError: Mountpoint must not already contain files

In [ ]:
# Colab-ready: Centralized experiments for Elliptic (LR, RF, XGB)
# - handles Drive mount safely
# - finds/merges Elliptic files (features+classes) or uses pre-merged file
# - runs seeded centralized experiments for seeds list
# - runs separate random runs if requested
# - prints results table in notebook and saves JSON + confusion matrix images
# NOTE: adjust paths/params at top where indicated.

import os, sys, json, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             f1_score, roc_auc_score, accuracy_score)
from sklearn.preprocessing import StandardScaler
import joblib
import warnings
warnings.filterwarnings("ignore")

# --------------------  USER CONFIG --------------------
DRIVE_ROOT = "/content/drive/MyDrive/PhD_FraudDetection"   # adjust if needed
DEFI_DATA_DIR = os.path.join(DRIVE_ROOT, "data", "defi")  # where you uploaded elliptic files
RESULTS_DIR = os.path.join(DRIVE_ROOT, "results_defi", "centralized")
SEED_LIST = [1011, 2022, 3033, 4044, 5055]
RANDOM_RUNS = 5   # if you want separate random runs (set to 0 to skip)
TEST_SIZE = 0.2
RANDOM_STATE_FOR_SPLIT = 42
TARGET_COL_NAMES_POSSIBLE = ["class", "is_fraud", "label", "isFraud"]  # possible label names
# -----------------------------------------------------

# 1) safe mount (only if not mounted)
try:
    from google.colab import drive
    if not os.path.exists('/content/drive') or len(os.listdir('/content/drive')) == 0:
        print("Mounting Google Drive...")
        drive.mount('/content/drive')
    else:
        print("Drive appears already mounted /content/drive (skipping mount).")
except Exception as e:
    print("Note: running outside Colab or drive mounting failed/was skipped:", str(e))

os.makedirs(RESULTS_DIR, exist_ok=True)

# 2) find dataset - prefer pre-merged file, else look for features+classes and merge
def find_and_load_elliptic(data_dir):
    # possible merged filenames
    merged_candidates = [
        os.path.join(data_dir, "elliptic_final_dataset.csv"),
        os.path.join(data_dir, "elliptic_final_dataset.csv".lower()),
        os.path.join(data_dir, "elliptic_final.csv"),
        os.path.join(data_dir, "defi_elliptic.csv"),
        os.path.join(data_dir, "elliptic_dataset.csv"),
        os.path.join(data_dir, "txs_merged.csv"),
    ]
    for f in merged_candidates:
        if os.path.exists(f):
            print("Found merged dataset:", f)
            return pd.read_csv(f)

    # search for features+classes CSV files anywhere under data_dir
    features = None
    classes = None
    for root, dirs, files in os.walk(data_dir):
        for fn in files:
            if "features" in fn.lower() and fn.lower().endswith(".csv"):
                features = os.path.join(root, fn)
            if ("classes" in fn.lower() or "txs_classes" in fn.lower()) and fn.lower().endswith(".csv"):
                classes = os.path.join(root, fn)
    if features and classes:
        print("Found features file:", features)
        print("Found classes file:", classes)
        df_feat = pd.read_csv(features)
        df_cls = pd.read_csv(classes)
        # common key: txId (Elliptic) or maybe "txId" column
        if "txId" in df_feat.columns and "txId" in df_cls.columns:
            merged = df_feat.merge(df_cls, on="txId", how="inner")
            print("Merged shape:", merged.shape)
            # save merged for future use
            outp = os.path.join(data_dir, "elliptic_final_dataset.csv")
            merged.to_csv(outp, index=False)
            print("Saved merged dataset to:", outp)
            return merged
        else:
            # attempt to detect common id column
            common = set(df_feat.columns).intersection(df_cls.columns)
            if len(common) > 0:
                key = list(common)[0]
                merged = df_feat.merge(df_cls, on=key, how="inner")
                print(f"Merged on {key} -> shape {merged.shape}")
                outp = os.path.join(data_dir, "elliptic_final_dataset.csv")
                merged.to_csv(outp, index=False)
                print("Saved merged dataset to:", outp)
                return merged
            else:
                raise FileNotFoundError("Couldn't find a shared join key between features and classes CSV.")
    # fallback: look for any csv in folder with many columns (maybe already uploaded)
    csvs = glob.glob(os.path.join(data_dir, "*.csv"))
    if csvs:
        # pick largest csv (by file size) as candidate
        csvs_sorted = sorted(csvs, key=lambda p: os.path.getsize(p), reverse=True)
        print("No explicit merged file found. Loading largest CSV candidate:", csvs_sorted[0])
        df = pd.read_csv(csvs_sorted[0])
        return df
    raise FileNotFoundError(f"No Elliptic files found in {data_dir}. Upload txs_features.csv + txs_classes.csv or a merged CSV.")

# load dataset
try:
    df = find_and_load_elliptic(DEFI_DATA_DIR)
except Exception as e:
    print("ERROR: Could not load dataset automatically:", e)
    print("Please upload txs_features.csv and txs_classes.csv into:", DEFI_DATA_DIR)
    raise

print("Loaded DataFrame shape:", df.shape)
print("Columns:", df.columns.tolist()[:40])

# 3) detect label column
label_col = None
for candidate in TARGET_COL_NAMES_POSSIBLE:
    if candidate in df.columns:
        label_col = candidate
        break
# try some common names
if label_col is None:
    # try 'class' or last column that looks like label
    if "class" in df.columns:
        label_col = "class"
    else:
        # if dataset has a small integer label column, try to detect it
        possible = [c for c in df.columns if df[c].nunique() <= 10]
        if possible:
            # heuristics: prefer 'class' or last such column
            label_col = possible[-1]
if label_col is None:
    raise RuntimeError("Could not detect label column automatically. Rename your label column to one of: " + ", ".join(TARGET_COL_NAMES_POSSIBLE))

print("Detected label column:", label_col)
print("Label distribution:\n", df[label_col].value_counts(dropna=False))

# For our experiments we need numerical features. We'll select a set of numeric columns.
# Elliptic features are many - pick numeric ones excluding id columns.
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# remove label and obvious ids
for c in [label_col, "txId", "TxHash"]:
    if c in numeric_cols:
        numeric_cols.remove(c)
# if no numeric features found, try other columns (convert if needed)
if len(numeric_cols) == 0:
    raise RuntimeError("No numeric feature columns detected. Inspect your CSV.")

print("Using numeric features (sample):", numeric_cols[:10])

# 4) function to train & evaluate centralized models for a given train/test split
def evaluate_model(y_true, y_pred_proba, threshold=0.5):
    # y_true are 0/1 labels; y_pred_proba are probabilities for class 1
    y_pred = (np.array(y_pred_proba) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = np.nan
    try:
        auc = roc_auc_score(y_true, y_pred_proba)
    except Exception:
        auc = float("nan")
    acc = accuracy_score(y_true, y_pred)
    return {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "precision": float(precision), "recall": float(recall),
            "f1": float(f1), "auc": float(auc), "accuracy": float(acc)}

def train_and_eval_centralized(X_train, y_train, X_test, y_test, out_prefix):
    # scaler
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    joblib.dump(scaler, out_prefix + "_scaler.pkl")

    results = {}
    ## Logistic Regression
    lr = LogisticRegression(max_iter=500, solver="liblinear")
    try:
        lr.fit(X_train_s, y_train)
        lr_probs = lr.predict_proba(X_test_s)[:,1]
    except Exception as e:
        print("LR training failed:", e)
        # fallback: all zeros probability
        lr_probs = np.zeros(len(y_test))
    results['LR'] = evaluate_model(y_test, lr_probs)
    joblib.dump(lr, out_prefix + "_lr.pkl")

    ## Random Forest
    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1)
    try:
        rf.fit(X_train, y_train)   # RF works with raw numeric features too
        rf_probs = rf.predict_proba(X_test)[:,1]
        results['RF'] = evaluate_model(y_test, rf_probs)
        joblib.dump(rf, out_prefix + "_rf.pkl")
    except Exception as e:
        print("RF training failed:", e)
        results['RF'] = evaluate_model(y_test, np.zeros(len(y_test)))

    ## XGBoost (if available)
    try:
        from xgboost import XGBClassifier
        xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', n_jobs=4)
        xgb.fit(X_train, y_train)
        xgb_probs = xgb.predict_proba(X_test)[:,1]
        results['XGB'] = evaluate_model(y_test, xgb_probs)
        joblib.dump(xgb, out_prefix + "_xgb.pkl")
    except Exception as e:
        print("XGB not available or failed:", e)
        results['XGB'] = evaluate_model(y_test, np.zeros(len(y_test)))

    # save confusion matrices as images
    for model_name, res in results.items():
        # reconstruct predicted labels from saved models if possible (for confusion image)
        # simpler: produce confusion matrix from predicted probabilities with threshold 0.5
        if model_name == 'LR':
            probs = lr_probs
        elif model_name == 'RF':
            probs = rf_probs
        else:
            probs = xgb_probs if 'xgb_probs' in locals() else np.zeros(len(y_test))
        preds = (np.array(probs) >= 0.5).astype(int)
        cm = confusion_matrix(y_test, preds, labels=[0,1])
        plt.figure(figsize=(4,4))
        plt.imshow(cm, interpolation='nearest')
        plt.title(f"Confusion matrix: {model_name}")
        plt.colorbar()
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, str(cm[i,j]), ha="center", va="center", color="white" if cm[i,j]>cm.max()/2 else "black")
        img_path = out_prefix + f"_{model_name}_confusion.png"
        pdf_path = out_prefix + f"_{model_name}_confusion.pdf"
        plt.tight_layout()
        plt.savefig(img_path)
        plt.savefig(pdf_path)
        plt.close()

    # persist metrics json
    with open(out_prefix + "_metrics.json", "w") as fh:
        json.dump(results, fh, indent=2)
    return results

# 5) Run experiments for each seed (seed-based centralized)
all_seed_results = []
for seed in SEED_LIST:
    print("\n=== Centralized run for seed:", seed, "===")
    # use deterministic split using seed
    # if label has single class across dataset (rare), we must warn
    if df[label_col].nunique() < 2:
        print("WARNING: label column has only one class. Cannot train supervised models.")
        break

    # convert label to 0/1 if not numeric
    y_series = df[label_col]
    if not pd.api.types.is_numeric_dtype(y_series):
        # try mapping '1','2','3' classes to fraud (elliptic uses class '1' for licit, '2' for illicit? check)
        # Here we map known Elliptic class names: '1' maybe licit, '2' illicit; user should inspect.
        # Default: map any value equal to '1' or 'licit' as 0, others as 1 (heuristic). Adjust if needed.
        unique_vals = list(y_series.unique())[:10]
        print("Non-numeric labels detected, sample unique:", unique_vals)
        # best effort mapping:
        val_map = {}
        # if classes contain strings 'licit'/'illicit' or '1'/'2', handle
        if any(str(x).lower().startswith("licit") for x in unique_vals):
            y_mapped = y_series.apply(lambda v: 0 if str(v).lower().startswith("licit") else 1)
        else:
            # try numeric cast
            try:
                y_mapped = pd.to_numeric(y_series, errors='coerce').fillna(0).astype(int)
                # if values are 1/2/3 etc, map >1 as fraud
                y_mapped = (y_mapped != y_mapped.min()).astype(int)
            except:
                y_mapped = (y_series != y_series.mode()[0]).astype(int)
        y = y_mapped.values
    else:
        # numeric labels: map to 0/1: minority -> 1
        ynum = df[label_col].astype(int)
        # if values are 0/1 OK
        if set(ynum.unique()).issubset({0,1}):
            y = ynum.values
        else:
            # map smallest value to 0, others to 1
            minv = ynum.min()
            y = (ynum != minv).astype(int).values

    X = df[numeric_cols].fillna(0).values

    # train/test split (stratify to keep label distribution)
    if len(np.unique(y)) > 1:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=seed, stratify=y)
    else:
        print("Only one class present after mapping. Aborting seed run.")
        break

    out_prefix = os.path.join(RESULTS_DIR, f"seed_{seed}", "seed"+str(seed))
    os.makedirs(os.path.dirname(out_prefix), exist_ok=True)
    res = train_and_eval_centralized(X_train, y_train, X_test, y_test, out_prefix)
    all_seed_results.append({"seed": seed, "results": res})
    # print summary row
    print("Seed", seed, "-> LR f1:", res['LR']['f1'], " RF f1:", res['RF']['f1'], " XGB f1:", res['XGB']['f1'])

# tabulate seed results in a DataFrame for display
rows = []
for entry in all_seed_results:
    s = entry['seed']
    for m, metrics in entry['results'].items():
        rows.append({ "seed": s, "model": m, **metrics })
if rows:
    df_seed_results = pd.DataFrame(rows)
    display(df_seed_results.sort_values(["model","seed"]).reset_index(drop=True))
    # save combined
    df_seed_results.to_csv(os.path.join(RESULTS_DIR, "combined_seeded_results.csv"), index=False)
    print("Saved combined seed results CSV to:", os.path.join(RESULTS_DIR, "combined_seeded_results.csv"))
else:
    print("No seed results were produced.")

# 6) Optional: separate random-run experiments (independent random splits)
if RANDOM_RUNS and RANDOM_RUNS > 0:
    print("\n=== Running separate random runs (n=", RANDOM_RUNS, ") ===")
    random_rows = []
    for r in range(1, RANDOM_RUNS+1):
        rs = np.random.randint(1, 10**6)
        print(f"\n-- Random run {r} (randstate={rs}) --")
        if len(np.unique(y)) > 1:
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=rs, stratify=y)
        else:
            print("Single-class labels, skipping random runs.")
            break
        out_prefix = os.path.join(RESULTS_DIR, f"random_run_{r}", f"run{r}")
        os.makedirs(os.path.dirname(out_prefix), exist_ok=True)
        res = train_and_eval_centralized(X_train, y_train, X_test, y_test, out_prefix)
        for m, metrics in res.items():
            random_rows.append({"run": r, "randstate": rs, "model": m, **metrics})
    if random_rows:
        df_random = pd.DataFrame(random_rows)
        display(df_random.sort_values(["model","run"]).reset_index(drop=True))
        df_random.to_csv(os.path.join(RESULTS_DIR, "combined_random_runs_results.csv"), index=False)
        print("Saved combined random run results CSV to:", os.path.join(RESULTS_DIR, "combined_random_runs_results.csv"))

print("\nAll centralized runs complete. Results saved under:", RESULTS_DIR)


Drive appears already mounted /content/drive (skipping mount).
ERROR: Could not load dataset automatically: No Elliptic files found in /content/drive/MyDrive/PhD_FraudDetection/data/defi. Upload txs_features.csv + txs_classes.csv or a merged CSV.
Please upload txs_features.csv and txs_classes.csv into: /content/drive/MyDrive/PhD_FraudDetection/data/defi


FileNotFoundError: No Elliptic files found in /content/drive/MyDrive/PhD_FraudDetection/data/defi. Upload txs_features.csv + txs_classes.csv or a merged CSV.

In [ ]:
!find /content/drive -name "*.csv" | head -20


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!find /content/drive/MyDrive -maxdepth 3 -type f \( -iname "*txs*" -o -iname "*elliptic*" \)


ValueError: Mountpoint must not already contain files

In [ ]:
!ls /content


drive  sample_data


In [ ]:
!find /content/drive/MyDrive -maxdepth 5 -type f \
    \( -iname "*elliptic*" -o -iname "*txs*" -o -iname "*.csv" \) \
    -printf "%p\n"


In [ ]:
from google.colab import files
uploaded = files.upload()

# This will open a file picker → select your Elliptic ZIP file


Saving Elliptic++ Dataset-20251207T172332Z-3-001.zip to Elliptic++ Dataset-20251207T172332Z-3-001.zip


In [ ]:
import zipfile
import os

zip_path = "Elliptic++ Dataset-20251207T172332Z-3-001.zip"  # change to your uploaded filename

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("elliptic_data")

print("Extracted files:")
os.listdir("elliptic_data")


Extracted files:


['Elliptic++ Dataset']

In [ ]:
import pandas as pd

features = "elliptic_data/elliptic_txs_features.csv"
classes = "elliptic_data/elliptic_txs_classes.csv"

df_feat = pd.read_csv(features)
df_class = pd.read_csv(classes)

print("Features:", df_feat.shape)
print("Classes:", df_class.shape)

df = df_feat.merge(df_class, on="txId", how="inner")
print("Merged Shape:", df.shape)
print(df["class"].value_counts())

df.to_csv("elliptic_final_dataset.csv", index=False)
print("Saved elliptic_final_dataset.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'elliptic_data/elliptic_txs_features.csv'

In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
except:
    print("Drive already mounted.")


Drive already mounted.


In [ ]:
!find /content/drive -maxdepth 5 -type f -iname "*elliptic*" -o -iname "*txs*"


In [ ]:
import zipfile

zip_path = list(uploaded.keys())[0]   # automatically picks the uploaded file

print("Extracting:", zip_path)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("elliptic_data")

print("Extracted to: elliptic_data/")


Extracting: Elliptic++ Dataset-20251207T172332Z-3-001.zip
Extracted to: elliptic_data/


In [ ]:
import os

for root, dirs, files in os.walk("elliptic_data"):
    for f in files:
        print(os.path.join(root, f))


elliptic_data/Elliptic++ Dataset/TxAddr_edgelist.csv
elliptic_data/Elliptic++ Dataset/txs_classes.csv
elliptic_data/Elliptic++ Dataset/txs_features.csv
elliptic_data/Elliptic++ Dataset/wallets_classes.csv
elliptic_data/Elliptic++ Dataset/AddrAddr_edgelist.csv
elliptic_data/Elliptic++ Dataset/AddrTx_edgelist.csv
elliptic_data/Elliptic++ Dataset/wallets_features_classes_combined.csv
elliptic_data/Elliptic++ Dataset/txs_edgelist.csv
elliptic_data/Elliptic++ Dataset/wallets_features.csv


In [ ]:
import pandas as pd

FEATURES = "elliptic_data/Elliptic++ Dataset/txs_features.csv"
CLASSES  = "elliptic_data/Elliptic++ Dataset/txs_classes.csv"

print("Loading features CSV...")
df_feat = pd.read_csv(FEATURES)
print("Features shape:", df_feat.shape)

print("Loading classes CSV...")
df_class = pd.read_csv(CLASSES)
print("Classes shape:", df_class.shape)

print("\nFeature columns:", df_feat.columns[:10])
print("Class columns:", df_class.columns)

# Merge using txId
df = df_feat.merge(df_class, on="txId", how="inner")

print("\nMerged dataset shape:", df.shape)
print(df["class"].value_counts())

# Save final dataset
df.to_csv("elliptic_merged.csv", index=False)
print("\nSaved as elliptic_merged.csv")


Loading features CSV...
Features shape: (203769, 184)
Loading classes CSV...
Classes shape: (203769, 2)

Feature columns: Index(['txId', 'Time step', 'Local_feature_1', 'Local_feature_2',
       'Local_feature_3', 'Local_feature_4', 'Local_feature_5',
       'Local_feature_6', 'Local_feature_7', 'Local_feature_8'],
      dtype='object')
Class columns: Index(['txId', 'class'], dtype='object')

Merged dataset shape: (203769, 185)
class
3    157205
2     42019
1      4545
Name: count, dtype: int64

Saved as elliptic_merged.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

print("Drive mounted.")

# Path of merged Elliptic dataset
DATA_PATH = "/content/elliptic_data/Elliptic++ Dataset/elliptic_merged.csv"

df = pd.read_csv(DATA_PATH)
print("Loaded:", df.shape)
print(df['class'].value_counts())

# Keep only known labels (1 = illicit, 2 = licit)
df = df[df["class"].isin([1, 2])]
df["label"] = df["class"].map({2: 0, 1: 1})   # 0 = normal, 1 = fraud
df = df.drop(columns=["class"])

print("After cleaning:", df.shape)
print(df["label"].value_counts())

# Features and label
X = df.drop(columns=["label", "txId"])
y = df["label"]

# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)


ValueError: Mountpoint must not already contain files

In [ ]:
import shutil

shutil.copy("/content/elliptic_merged.csv",
            "/content/drive/MyDrive/PhD_FraudDetection/data/defi/elliptic_merged.csv")

print("Copied to Drive!")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/PhD_FraudDetection/data/defi/elliptic_merged.csv'

In [ ]:
!find /content -maxdepth 3 -type f -name "elliptic_merged.csv"
!find /content/drive -maxdepth 5 -type f -name "elliptic_merged.csv"


/content/elliptic_merged.csv


In [ ]:
# ================================================
# CENTRALIZED TRAINING ON ELLIPTIC DATASET
# LR, RF, XGB  — Seed-Based + Random Runs
# ================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import json
import os

DATA_PATH = "/content/elliptic_merged.csv"
OUTPUT = "/content/centralized_elliptic_results"
os.makedirs(OUTPUT, exist_ok=True)

print("Loading dataset...")
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)

# --------------------------------------------
# Prepare features & binary labels
# class: 1 = illicit, 2 = licit, 3 = unknown
# For fraud detection:
# fraudulent = class == 1
# --------------------------------------------

df["label"] = df["class"].apply(lambda x: 1 if x == 1 else 0)
df = df.drop(columns=["class", "txId"])

y = df["label"].values
X = df.drop(columns=["label"]).values

print("Fraud distribution:")
print(pd.Series(y).value_counts())

# ============================================
# Shared train/test split (fixed for all seeds)
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ============================================
# Helper: Evaluate model
# ============================================

def evaluate(model_name, y_true, y_pred, y_prob):
    return {
        "model": model_name,
        "tn": int(confusion_matrix(y_true, y_pred)[0][0]),
        "fp": int(confusion_matrix(y_true, y_pred)[0][1]),
        "fn": int(confusion_matrix(y_true, y_pred)[1][0]),
        "tp": int(confusion_matrix(y_true, y_pred)[1][1]),
        "precision": float(np.round(classification_report(y_true, y_pred, output_dict=True)["1"]["precision"], 4)),
        "recall": float(np.round(classification_report(y_true, y_pred, output_dict=True)["1"]["recall"], 4)),
        "f1": float(np.round(classification_report(y_true, y_pred, output_dict=True)["1"]["f1-score"], 4)),
        "auc": float(np.round(roc_auc_score(y_true, y_prob), 4)),
        "accuracy": float(np.round((y_pred == y_true).mean(), 4)),
    }

# ============================================
# MODELS
# ============================================

def get_models(seed):
    return {
        "LR": LogisticRegression(max_iter=500, random_state=seed),
        "RF": RandomForestClassifier(n_estimators=300, max_depth=None, random_state=seed),
        "XGB": XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            random_state=seed,
            eval_metric="logloss"
        )
    }

# ============================================
# SEED-BASED EXPERIMENTS
# ============================================
SEEDS = [1011, 2022, 3033, 4044, 5055]
seed_results = []

for seed in SEEDS:
    print(f"\n==============================")
    print(f" Running SEED: {seed}")
    print("==============================")

    models = get_models(seed)
    results = {}

    for name, model in models.items():
        print(f"\nTraining {name}...")
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]

        metrics = evaluate(name, y_test, preds, probs)
        results[name] = metrics

        # Save metrics JSON
        with open(f"{OUTPUT}/metrics_seed_{seed}_{name}.json", "w") as f:
            json.dump(metrics, f, indent=4)

    seed_results.append(results)

print("\nSeed-based experiments completed.")

# ============================================
# RANDOM-RUN EXPERIMENTS  (5 runs)
# ============================================
RANDOM_RUNS = 5
random_results = []

for run in range(1, RANDOM_RUNS + 1):
    print(f"\n==============================")
    print(f" Random Run {run}")
    print("==============================")

    seed = np.random.randint(1, 99999)
    models = get_models(seed)
    results = {}

    for name, model in models.items():
        print(f"\nTraining {name}...")
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]

        metrics = evaluate(name, y_test, preds, probs)
        results[name] = metrics

        # Save metrics JSON
        with open(f"{OUTPUT}/metrics_random_run_{run}_{name}.json", "w") as f:
            json.dump(metrics, f, indent=4)

    random_results.append(results)

print("\nRandom-run experiments completed.")

# ============================================
# SAVE COMBINED RESULTS
# ============================================

with open(f"{OUTPUT}/all_seed_results.json", "w") as f:
    json.dump(seed_results, f, indent=4)

with open(f"{OUTPUT}/all_random_results.json", "w") as f:
    json.dump(random_results, f, indent=4)

print("\n\n✅ CENTRALIZED EXPERIMENTS COMPLETED")
print(f"Results saved in: {OUTPUT}")


Loading dataset...
Shape: (203769, 185)
Fraud distribution:
0    199224
1      4545
Name: count, dtype: int64

 Running SEED: 1011

Training LR...


ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load merged dataset
df = pd.read_csv("/content/elliptic_merged.csv")
print("Loaded:", df.shape)

# Keep only numeric columns
df = df.select_dtypes(include=[np.number])

# Replace class labels:
# 1 = Fraud, 2 & 3 = Non-fraud
df['label'] = df['class'].apply(lambda x: 1 if x == 1 else 0)
df = df.drop(columns=['class'])

print("Final columns:", df.columns)
print("\nLabel counts:\n", df['label'].value_counts())

# ---- Handle missing values (VERY IMPORTANT) ----
df.fillna(df.mean(), inplace=True)
print("\nNaN after fill:", df.isna().sum().sum())

# ---- Split into train/test ----
X = df.drop(columns=['label'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---- Scale features ----
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print("\nTrain:", X_train.shape, " Test:", X_test.shape)


Loaded: (203769, 185)
Final columns: Index(['txId', 'Time step', 'Local_feature_1', 'Local_feature_2',
       'Local_feature_3', 'Local_feature_4', 'Local_feature_5',
       'Local_feature_6', 'Local_feature_7', 'Local_feature_8',
       ...
       'in_BTC_max', 'in_BTC_mean', 'in_BTC_median', 'in_BTC_total',
       'out_BTC_min', 'out_BTC_max', 'out_BTC_mean', 'out_BTC_median',
       'out_BTC_total', 'label'],
      dtype='object', length=185)

Label counts:
 label
0    199224
1      4545
Name: count, dtype: int64

NaN after fill: 0

Train: (163015, 184)  Test: (40754, 184)


In [ ]:
# ============================================
# CENTRALIZED DEFI (ELLIPTIC)
# Seed-Based Experiments for LR, RF, XGB
# ============================================

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
from sklearn.metrics import confusion_matrix

import json
import os

# --------------------------------------------
# Use the cleaned df and the train/test split
# --------------------------------------------
print("Centralized DeFi Experiments Starting...")
print("Train:", X_train.shape, " Test:", X_test.shape)

# --------------------------------------------
# Function to compute metrics
# --------------------------------------------
def compute_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "precision": round(tp / (tp + fp + 1e-8), 4),
        "recall": round(tp / (tp + fn + 1e-8), 4),
        "f1": round(2 * tp / (2 * tp + fp + fn + 1e-8), 4),
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "auc": round(roc_auc_score(y_true, y_prob), 4)
    }

# --------------------------------------------
# Models
# --------------------------------------------
def get_models(seed):
    return {
        "LR": LogisticRegression(max_iter=500, class_weight="balanced", solver="liblinear"),
        "RF": RandomForestClassifier(n_estimators=300, random_state=seed, class_weight="balanced"),
        "XGB": XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            random_state=seed
        )
    }

# --------------------------------------------
# Directory for results
# --------------------------------------------
out_dir = "/content/centralized_elliptic_results"
os.makedirs(out_dir, exist_ok=True)

SEEDS = [1011, 2022, 3033, 4044, 5055]

all_results = {}

# --------------------------------------------
# Run experiments
# --------------------------------------------
for seed in SEEDS:
    print("\n==============================")
    print(f" Running Seed {seed}")
    print("==============================")

    np.random.seed(seed)

    models = get_models(seed)

    seed_result = {}

    for name, model in models.items():
        print(f"\nTraining {name}...")

        model.fit(X_train, y_train)
        prob = model.predict_proba(X_test)[:, 1]

        metrics = compute_metrics(y_test, prob)
        seed_result[name] = metrics

        print(f"{name} Metrics:", metrics)

    all_results[seed] = seed_result

# Save full results JSON
with open(os.path.join(out_dir, "centralized_seed_results.json"), "w") as f:
    json.dump(all_results, f, indent=4)

print("\n✓ Centralized Seed-Based Experiments Completed!")
print("Results saved to:", out_dir)


Centralized DeFi Experiments Starting...
Train: (163015, 184)  Test: (40754, 184)

 Running Seed 1011

Training LR...
LR Metrics: {'tn': 34467, 'fp': 5378, 'fn': 97, 'tp': 812, 'precision': np.float64(0.1312), 'recall': np.float64(0.8933), 'f1': np.float64(0.2288), 'accuracy': 0.8657, 'auc': np.float64(0.9325)}

Training RF...
RF Metrics: {'tn': 39783, 'fp': 62, 'fn': 224, 'tp': 685, 'precision': np.float64(0.917), 'recall': np.float64(0.7536), 'f1': np.float64(0.8273), 'accuracy': 0.993, 'auc': np.float64(0.9941)}

Training XGB...
XGB Metrics: {'tn': 39762, 'fp': 83, 'fn': 163, 'tp': 746, 'precision': np.float64(0.8999), 'recall': np.float64(0.8207), 'f1': np.float64(0.8585), 'accuracy': 0.994, 'auc': np.float64(0.9943)}

 Running Seed 2022

Training LR...
LR Metrics: {'tn': 34467, 'fp': 5378, 'fn': 97, 'tp': 812, 'precision': np.float64(0.1312), 'recall': np.float64(0.8933), 'f1': np.float64(0.2288), 'accuracy': 0.8657, 'auc': np.float64(0.9325)}

Training RF...


In [ ]:
import pandas as pd

# If loading from Drive:
# df = pd.read_csv("/content/drive/MyDrive/...")

# If uploaded:
# df = pd.read_csv("your_file.csv")

df.head()          # Show first rows
df.columns.tolist()  # Show all column names


NameError: name 'df' is not defined

In [ ]:
# ============================================
#   STEP 1: MOUNT GOOGLE DRIVE
# ============================================
from google.colab import drive
drive.mount('/content/drive')

print("Drive mounted successfully.")


Mounted at /content/drive
Drive mounted successfully.


In [ ]:
# ============================================
#   STEP 2: CHECK YOUR DATASET LOCATION
# ============================================
import os

base_path = "/content/drive/MyDrive/"
for root, dirs, files in os.walk(base_path):
    for file in files:
        if "elliptic" in file.lower():
            print("Found:", os.path.join(root, file))


Found: /content/drive/MyDrive/Colab Notebooks/DEFI + Elliptic dataset .ipynb
Found: /content/drive/MyDrive/PhD_FraudDetection/data/defi/elliptic_merged.csv


In [ ]:
# ===============================
# DEFi (Elliptic) — Colab-ready
# Two-stage pipeline:
#  - Stage 1: Seed-based (once per seed per model)
#  - Stage 2: Random-runs (5 runs)
# Models: LogisticRegression, RandomForest, XGBoost
# Federation: FedAvg for LR; prediction-averaging (soft voting) for RF/XGB
# ===============================

# ---------- Imports ----------
import os
import random
from datetime import datetime
import numpy as np
import pandas as pd
from collections import Counter

# sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
# xgboost
import xgboost as xgb

# For plotting (optional)
import matplotlib.pyplot as plt

# ---------- Settings ----------
DATA_PATH = "/content/drive/MyDrive/PhD_FraudDetection/data/defi/elliptic_merged.csv"
RESULTS_DIR = "/content/drive/MyDrive/PhD_FraudDetection/results_defi"
os.makedirs(RESULTS_DIR, exist_ok=True)

seed_list = [1011, 2022, 3033, 4044, 5055]
num_random_runs = 5
num_clients = 5

# Model hyperparams (tune as needed)
rf_params = {"n_estimators": 150, "n_jobs": -1}
xgb_params = {"use_label_encoder": False, "eval_metric": "logloss", "n_estimators":150, "verbosity":0}
lr_params = {"max_iter": 1000, "solver": "lbfgs"}

# ---------- Utility functions ----------
def print_banner(msg):
    print("\n" + "="*8 + " " + msg + " " + "="*8 + "\n")

def compute_metrics(y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_proba)
    except Exception:
        auc = np.nan
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {"Accuracy": acc, "Precision": prec, "Recall": rec,
            "F1": f1, "AUC": auc, "TP": int(tp), "TN": int(tn),
            "FP": int(fp), "FN": int(fn)}

def stratified_client_split(X, y, n_clients=5, random_state=None):
    # returns list of (X_client, y_client)
    # We'll use StratifiedKFold to partition into n_clients folds
    skf = StratifiedKFold(n_splits=n_clients, shuffle=True, random_state=random_state)
    clients = []
    for _, idx in skf.split(X, y):
        clients.append((X[idx], y.iloc[idx] if isinstance(y, pd.Series) else y[idx]))
    return clients

def average_lr_weights(weights_list):
    # weights_list: list of tuples (coef, intercept)
    coefs = np.array([w[0] for w in weights_list])
    inters = np.array([w[1] for w in weights_list])
    return np.mean(coefs, axis=0), np.mean(inters, axis=0)

# ---------- Load dataset and inspect label column ----------
print_banner("LOAD DATA")
df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
print("Columns preview:", df.columns.tolist()[:40])

# Attempt to detect label column from common names
possible_labels = ["class", "label", "is_illicit", "fraud", "isFraud"]
label_col = None
for cand in possible_labels:
    if cand in df.columns:
        label_col = cand
        break

# If not found, try to guess by small heuristics: column with small number of unique values (2 or 3)
if label_col is None:
    for c in df.columns:
        if df[c].nunique() <= 3 and df[c].dtype in [np.int64, np.int32, np.uint8, object]:
            # skip id-like columns
            if c.lower().startswith("tx") or c.lower().startswith("id"):
                continue
            label_col = c
            break

if label_col is None:
    raise RuntimeError("Couldn't detect label column automatically. Please open the dataset and tell me the label column name.")

print("Detected label column:", label_col)
# Show class balance
print("Class counts:\n", df[label_col].value_counts())

# ---------- Prepare features and labels ----------
# Drop non-feature columns: typical columns to drop if present
drop_candidates = ["txId", "transactionId", "time", "timestamp", "step", "node"]
X = df.drop(columns=[label_col] + [c for c in drop_candidates if c in df.columns], errors='ignore')
y = df[label_col].astype(int)  # ensure integer labels

# If there are non-numeric columns, try encode or drop (simple approach)
non_numeric = X.select_dtypes(include=['object', 'category']).columns.tolist()
if non_numeric:
    print("Non-numeric feature columns detected (will be dropped):", non_numeric)
    X = X.drop(columns=non_numeric)

# Convert to numpy for splits later
X_values = X.values
print("Final feature matrix shape:", X_values.shape)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_values)

# ---------- Define base model factory ----------
def make_models(random_state):
    lr = LogisticRegression(random_state=random_state, **lr_params)
    rf = RandomForestClassifier(random_state=random_state, **rf_params)
    xgb_clf = xgb.XGBClassifier(random_state=random_state, **xgb_params)
    return {"LR": lr, "RF": rf, "XGB": xgb_clf}

# ---------- Stage 1: Seed-based runs ----------
print_banner("STAGE 1: SEED-BASED RUNS")

stage1_centralized_results = []
stage1_fl_results = []

for seed in seed_list:
    print_banner(f"Seed {seed} — CENTRALIZED")
    models = make_models(random_state=seed)
    # central train/test split per seed (once)
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed)

    # Train centralized models once per seed
    for m_name, model in models.items():
        # set random_state where applicable
        if hasattr(model, "random_state"):
            try:
                model.random_state = seed
            except:
                pass
        model.fit(X_train, y_train)
        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X_test)[:,1]
        else:
            # fallback to decision_function
            try:
                y_proba = model.decision_function(X_test)
            except:
                y_proba = np.zeros(len(y_test))
        y_pred = model.predict(X_test)
        metrics = compute_metrics(y_test, y_pred, y_proba)
        metrics.update({"Seed": seed, "RunType": "Seed-based", "Stage": "Centralized", "Model": m_name})
        stage1_centralized_results.append(metrics)
        print(f"[CENT] Seed={seed}, Model={m_name} -> Acc={metrics['Accuracy']:.4f}, F1={metrics['F1']:.4f}")

    # ---------- Federated for this seed ----------
    print_banner(f"Seed {seed} — FEDERATED")
    # Prepare stratified client splits using the same seed for reproducibility
    clients = stratified_client_split(X_scaled, y, n_clients=num_clients, random_state=seed)

    # For LR we will collect local weights for FedAvg
    # For RF/XGB we will do soft-voting aggregation of probabilities

    # LOCAL TRAINING
    local_models = {"LR": [], "RF": [], "XGB": []}
    for idx, (Xc, yc) in enumerate(clients):
        client_rs = int(seed + idx + 1)  # deterministic client random-state
        cm = make_models(random_state=client_rs)
        for name, model in cm.items():
            model.fit(Xc, yc)
            local_models[name].append(model)
        print(f"Trained client {idx+1}/{len(clients)}")

    # AGGREGATE AND EVALUATE
    for m_name in ["LR", "RF", "XGB"]:
        if m_name == "LR":
            # get weights from local LR models
            weights = []
            for lm in local_models["LR"]:
                weights.append((lm.coef_.copy(), lm.intercept_.copy()))
            avg_coef, avg_intercept = average_lr_weights(weights)
            # build a fresh lr with same shape and set weights
            global_lr = LogisticRegression(**lr_params)
            # We need to fit it to set attributes shape; fit on a small batch:
            global_lr.fit(X_train[:10], y_train[:10])
            global_lr.coef_ = avg_coef
            global_lr.intercept_ = avg_intercept
            # For sklearn logistic, also set classes_
            global_lr.classes_ = np.unique(y)
            # Evaluate
            y_proba = global_lr.predict_proba(X_test)[:,1]
            y_pred = global_lr.predict(X_test)
        else:
            # Soft-voting average of client predicted probabilities
            prob_preds = []
            for lm in local_models[m_name]:
                if hasattr(lm, "predict_proba"):
                    prob_preds.append(lm.predict_proba(X_test)[:,1])
                else:
                    # fallback
                    prob_preds.append(np.zeros(len(X_test)))
            avg_proba = np.mean(np.vstack(prob_preds), axis=0)
            y_proba = avg_proba
            y_pred = (avg_proba >= 0.5).astype(int)

        metrics = compute_metrics(y_test, y_pred, y_proba)
        metrics.update({"Seed": seed, "RunType": "Seed-based", "Stage": "Federated", "Model": m_name})
        stage1_fl_results.append(metrics)
        print(f"[FL] Seed={seed}, Model={m_name} -> Acc={metrics['Accuracy']:.4f}, F1={metrics['F1']:.4f}")

# Save Stage1 results
stage1_cent_df = pd.DataFrame(stage1_centralized_results)
stage1_fl_df = pd.DataFrame(stage1_fl_results)
stage1_cent_df.to_csv(os.path.join(RESULTS_DIR, "stage1_centralized_seed_based.csv"), index=False)
stage1_fl_df.to_csv(os.path.join(RESULTS_DIR, "stage1_federated_seed_based.csv"), index=False)
print("Stage 1 results saved at:", RESULTS_DIR)

# ---------- Stage 2: Random runs ----------
print_banner("STAGE 2: RANDOM-RUNS")
stage2_centralized_results = []
stage2_fl_results = []

for run_num in range(1, num_random_runs+1):
    # choose a random seed per run for reproducibility logging but use random_state=None in splits to create randomness
    run_seed = int(datetime.now().timestamp()) % 100000 + run_num
    print_banner(f"Random Run {run_num} (seed hint {run_seed}) — CENTRALIZED")
    models = make_models(random_state=run_seed)

    # We'll create a new randomized split each run (no fixed seed), but to keep reproducible you could set random_state=run_seed
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=run_seed)

    for m_name, model in models.items():
        model.fit(X_train, y_train)
        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X_test)[:,1]
        else:
            try:
                y_proba = model.decision_function(X_test)
            except:
                y_proba = np.zeros(len(y_test))
        y_pred = model.predict(X_test)
        metrics = compute_metrics(y_test, y_pred, y_proba)
        metrics.update({"RandomRun": run_num, "RunType": "Random-run", "Stage": "Centralized", "Model": m_name})
        stage2_centralized_results.append(metrics)
        print(f"[CENT] Run={run_num}, Model={m_name} -> Acc={metrics['Accuracy']:.4f}, F1={metrics['F1']:.4f}")

    # ---------- Federated for this random run ----------
    print_banner(f"Random Run {run_num} — FEDERATED")
    # Prepare stratified client splits (use run_seed for reproducibility)
    clients = stratified_client_split(X_scaled, y, n_clients=num_clients, random_state=run_seed)

    # Train local models on clients
    local_models = {"LR": [], "RF": [], "XGB": []}
    for idx, (Xc, yc) in enumerate(clients):
        client_rs = run_seed + idx + 1
        cm = make_models(random_state=client_rs)
        for name, model in cm.items():
            model.fit(Xc, yc)
            local_models[name].append(model)
        print(f"Trained client {idx+1}/{len(clients)} for run {run_num}")

    # Aggregate and evaluate
    for m_name in ["LR", "RF", "XGB"]:
        if m_name == "LR":
            weights = []
            for lm in local_models["LR"]:
                weights.append((lm.coef_.copy(), lm.intercept_.copy()))
            avg_coef, avg_intercept = average_lr_weights(weights)
            global_lr = LogisticRegression(**lr_params)
            global_lr.fit(X_train[:10], y_train[:10])
            global_lr.coef_ = avg_coef
            global_lr.intercept_ = avg_intercept
            global_lr.classes_ = np.unique(y)
            y_proba = global_lr.predict_proba(X_test)[:,1]
            y_pred = global_lr.predict(X_test)
        else:
            prob_preds = []
            for lm in local_models[m_name]:
                if hasattr(lm, "predict_proba"):
                    prob_preds.append(lm.predict_proba(X_test)[:,1])
                else:
                    prob_preds.append(np.zeros(len(X_test)))
            avg_proba = np.mean(np.vstack(prob_preds), axis=0)
            y_proba = avg_proba
            y_pred = (avg_proba >= 0.5).astype(int)

        metrics = compute_metrics(y_test, y_pred, y_proba)
        metrics.update({"RandomRun": run_num, "RunType": "Random-run", "Stage": "Federated", "Model": m_name})
        stage2_fl_results.append(metrics)
        print(f"[FL] Run={run_num}, Model={m_name} -> Acc={metrics['Accuracy']:.4f}, F1={metrics['F1']:.4f}")

# Save Stage2 results
stage2_cent_df = pd.DataFrame(stage2_centralized_results)
stage2_fl_df = pd.DataFrame(stage2_fl_results)
stage2_cent_df.to_csv(os.path.join(RESULTS_DIR, "stage2_centralized_random_runs.csv"), index=False)
stage2_fl_df.to_csv(os.path.join(RESULTS_DIR, "stage2_federated_random_runs.csv"), index=False)
print("Stage 2 results saved at:", RESULTS_DIR)

# ---------- Combined summaries ----------
print_banner("SUMMARY SAVES")
all_cent = pd.concat([stage1_cent_df, stage2_cent_df], ignore_index=True)
all_fl = pd.concat([stage1_fl_df, stage2_fl_df], ignore_index=True)
all_cent.to_csv(os.path.join(RESULTS_DIR, "all_centralized_results.csv"), index=False)
all_fl.to_csv(os.path.join(RESULTS_DIR, "all_federated_results.csv"), index=False)
print("Combined CSVs saved.")

# Optional: quick plot of F1 vs Model for seed-based centralized
try:
    import seaborn as sns
    sns.set(style="whitegrid")
    fig, ax = plt.subplots(figsize=(8,4))
    sns.barplot(data=stage1_cent_df, x="Model", y="F1", ax=ax)
    ax.set_title("Stage1: Seed-based Centralized F1 by Model (avg across seeds)")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "stage1_centralized_f1_by_model.png"))
    plt.close(fig)
    print("Saved quick plot: stage1_centralized_f1_by_model.png")
except Exception as e:
    print("Plotting skipped (seaborn may not be installed).", str(e))

print_banner("DONE — Stage1 + Stage2 Completed (Centralized + FL).")
print("Check folder:", RESULTS_DIR)



======== LOAD DATA ========

Loaded: /content/drive/MyDrive/PhD_FraudDetection/data/defi/elliptic_merged.csv
Shape: (203769, 185)
Columns preview: ['txId', 'Time step', 'Local_feature_1', 'Local_feature_2', 'Local_feature_3', 'Local_feature_4', 'Local_feature_5', 'Local_feature_6', 'Local_feature_7', 'Local_feature_8', 'Local_feature_9', 'Local_feature_10', 'Local_feature_11', 'Local_feature_12', 'Local_feature_13', 'Local_feature_14', 'Local_feature_15', 'Local_feature_16', 'Local_feature_17', 'Local_feature_18', 'Local_feature_19', 'Local_feature_20', 'Local_feature_21', 'Local_feature_22', 'Local_feature_23', 'Local_feature_24', 'Local_feature_25', 'Local_feature_26', 'Local_feature_27', 'Local_feature_28', 'Local_feature_29', 'Local_feature_30', 'Local_feature_31', 'Local_feature_32', 'Local_feature_33', 'Local_feature_34', 'Local_feature_35', 'Local_feature_36', 'Local_feature_37', 'Local_feature_38']
Detected label column: class
Class counts:
 class
3    157205
2     42019
1    

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
# ---------- Prepare features and labels ----------
# Drop non-feature columns: typical columns to drop if present
drop_candidates = ["txId", "transactionId", "time", "timestamp", "step", "node", "Time step"]
X = df.drop(columns=[label_col] + [c for c in drop_candidates if c in df.columns], errors='ignore')
y = df[label_col].astype(int)

# Detect non-numeric columns and drop them (Elliptic usually has none)
non_numeric = X.select_dtypes(include=['object', 'category']).columns.tolist()
if non_numeric:
    print("Dropping non-numeric columns:", non_numeric)
    X = X.drop(columns=non_numeric)

# Convert to numpy
X_values = X.values
print("Before imputation — NaNs:", np.isnan(X_values).sum())

# ===== FIX: IMPUTE MISSING VALUES =====
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="mean")  # fills NaN with column-wise mean
X_values = imputer.fit_transform(X_values)

print("After imputation — NaNs:", np.isnan(X_values).sum())

# Scale features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_values)

print("Final feature matrix shape:", X_scaled.shape)


Before imputation — NaNs: 16405
After imputation — NaNs: 0
Final feature matrix shape: (203769, 182)


In [ ]:
# ===============================
# DEFI (Elliptic) — FINAL WORKING CODE
# With NaN Imputation FIX
# Stage 1: Seed-Based (Centralized + FL)
# Stage 2: Random Runs (Centralized + FL)
# ===============================

import os
import random
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import xgboost as xgb

# ===========================================================
# CONFIGURATION
# ===========================================================
DATA_PATH = "/content/drive/MyDrive/PhD_FraudDetection/data/defi/elliptic_merged.csv"
RESULTS_DIR = "/content/drive/MyDrive/PhD_FraudDetection/results_defi"
os.makedirs(RESULTS_DIR, exist_ok=True)

seed_list = [1011, 2022, 3033, 4044, 5055]
num_random_runs = 5
num_clients = 5

rf_params = {"n_estimators": 150, "n_jobs": -1}
xgb_params = {"eval_metric": "logloss", "verbosity": 0, "n_estimators": 150}
lr_params = {"max_iter": 1000}

# ===========================================================
# HELPER FUNCTIONS
# ===========================================================
def compute_metrics(y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_proba)
    except:
        auc = np.nan
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1,
            "AUC": auc, "TP": tp, "TN": tn, "FP": fp, "FN": fn}

def stratified_client_split(X, y, n_clients=5, random_state=None):
    skf = StratifiedKFold(n_splits=n_clients, shuffle=True, random_state=random_state)
    clients = []
    for _, idx in skf.split(X, y):
        clients.append((X[idx], y.iloc[idx]))
    return clients

def average_lr_weights(weights_list):
    coefs = np.array([w[0] for w in weights_list])
    inters = np.array([w[1] for w in weights_list])
    return np.mean(coefs, axis=0), np.mean(inters, axis=0)

# ===========================================================
# LOAD DATA
# ===========================================================
print("Loading dataset...")
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)

# Detect label column
label_col = "class"
print("Detected label column:", label_col)
print("Class counts:\n", df[label_col].value_counts())

# ===========================================================
# PREPARE FEATURES (WITH NAN FIX)
# ===========================================================
drop_cols = ["txId", "Time step", "time", "timestamp", "transactionId", "step", "node"]
X = df.drop(columns=[label_col] + [c for c in drop_cols if c in df.columns], errors='ignore')
y = df[label_col].astype(int)

# Remove non-numeric if any
non_numeric = X.select_dtypes(include=['object']).columns.tolist()
if non_numeric:
    X = X.drop(columns=non_numeric)

X_values = X.values

# --- FIX NaN VALUES ---
imputer = SimpleImputer(strategy="mean")
X_values = imputer.fit_transform(X_values)
print("NaN after imputation:", np.isnan(X_values).sum())

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_values)

# Model factory
def make_models(random_state):
    return {
        "LR": LogisticRegression(random_state=random_state, **lr_params),
        "RF": RandomForestClassifier(random_state=random_state, **rf_params),
        "XGB": xgb.XGBClassifier(random_state=random_state, **xgb_params),
    }

# ===========================================================
# STAGE 1 — SEED BASED RUNS
# ===========================================================
stage1_cent = []
stage1_fl = []

print("\n=========== STAGE 1: SEED-BASED ===========")

for seed in seed_list:

    print(f"\n--- SEED {seed} (CENTRALIZED) ---")
    models = make_models(seed)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        m = compute_metrics(y_test, y_pred, y_proba)
        m.update({"Seed": seed, "Model": name, "Stage": "Centralized"})
        stage1_cent.append(m)
        print(f"[CENT] Seed={seed}, Model={name}, F1={m['F1']:.4f}")

    print(f"\n--- SEED {seed} (FEDERATED) ---")

    clients = stratified_client_split(X_scaled, y, num_clients, seed)

    local_models = {"LR": [], "RF": [], "XGB": []}

    # Local training
    for i, (Xc, yc) in enumerate(clients):
        cm = make_models(seed + i)
        for name, model in cm.items():
            model.fit(Xc, yc)
            local_models[name].append(model)

    # Aggregation and evaluation
    for name in ["LR", "RF", "XGB"]:
        if name == "LR":
            weights = [(lm.coef_.copy(), lm.intercept_.copy()) for lm in local_models["LR"]]
            coef, inter = average_lr_weights(weights)
            global_lr = LogisticRegression(**lr_params)
            global_lr.fit(X_train[:10], y_train[:10])
            global_lr.coef_ = coef
            global_lr.intercept_ = inter
            global_lr.classes_ = np.unique(y)
            y_proba = global_lr.predict_proba(X_test)[:, 1]
            y_pred = global_lr.predict(X_test)
        else:
            probs = [lm.predict_proba(X_test)[:, 1] for lm in local_models[name]]
            avg_proba = np.mean(np.vstack(probs), axis=0)
            y_proba = avg_proba
            y_pred = (avg_proba >= 0.5).astype(int)

        m = compute_metrics(y_test, y_pred, y_proba)
        m.update({"Seed": seed, "Model": name, "Stage": "Federated"})
        stage1_fl.append(m)
        print(f"[FL] Seed={seed}, Model={name}, F1={m['F1']:.4f}")

# Save Stage 1 results
pd.DataFrame(stage1_cent).to_csv(f"{RESULTS_DIR}/stage1_centralized_seed.csv", index=False)
pd.DataFrame(stage1_fl).to_csv(f"{RESULTS_DIR}/stage1_federated_seed.csv", index=False)

# ===========================================================
# STAGE 2 — RANDOM RUNS
# ===========================================================
stage2_cent = []
stage2_fl = []

print("\n=========== STAGE 2: RANDOM RUNS ===========")

for run in range(1, num_random_runs + 1):

    print(f"\n--- RANDOM RUN {run} (CENTRALIZED) ---")

    rseed = 1000 + run
    models = make_models(rseed)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=rseed
    )

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        m = compute_metrics(y_test, y_pred, y_proba)
        m.update({"RandomRun": run, "Model": name, "Stage": "Centralized"})
        stage2_cent.append(m)
        print(f"[CENT] Run={run}, Model={name}, F1={m['F1']:.4f}")

    print(f"\n--- RANDOM RUN {run} (FEDERATED) ---")

    clients = stratified_client_split(X_scaled, y, num_clients, rseed)

    local_models = {"LR": [], "RF": [], "XGB": []}

    for i, (Xc, yc) in enumerate(clients):
        cm = make_models(rseed + i)
        for name, model in cm.items():
            model.fit(Xc, yc)
            local_models[name].append(model)

    for name in ["LR", "RF", "XGB"]:
        if name == "LR":
            weights = [(lm.coef_.copy(), lm.intercept_.copy()) for lm in local_models["LR"]]
            coef, inter = average_lr_weights(weights)
            global_lr = LogisticRegression(**lr_params)
            global_lr.fit(X_train[:10], y_train[:10])
            global_lr.coef_ = coef
            global_lr.intercept_ = inter
            global_lr.classes_ = np.unique(y)
            y_proba = global_lr.predict_proba(X_test)[:, 1]
            y_pred = global_lr.predict(X_test)
        else:
            probs = [lm.predict_proba(X_test)[:, 1] for lm in local_models[name]]
            avg_proba = np.mean(np.vstack(probs), axis=0)
            y_proba = avg_proba
            y_pred = (avg_proba >= 0.5).astype(int)

        m = compute_metrics(y_test, y_pred, y_proba)
        m.update({"RandomRun": run, "Model": name, "Stage": "Federated"})
        stage2_fl.append(m)
        print(f"[FL] Run={run}, Model={name}, F1={m['F1']:.4f}")

# Save Stage 2 results
pd.DataFrame(stage2_cent).to_csv(f"{RESULTS_DIR}/stage2_centralized_random.csv", index=False)
pd.DataFrame(stage2_fl).to_csv(f"{RESULTS_DIR}/stage2_federated_random.csv", index=False)

print("\n============= ALL DONE =============")
print("Results saved in:", RESULTS_DIR)


Loading dataset...
Shape: (203769, 185)
Detected label column: class
Class counts:
 class
3    157205
2     42019
1      4545
Name: count, dtype: int64
NaN after imputation: 0

=========== STAGE 1: SEED-BASED ===========

--- SEED 1011 (CENTRALIZED) ---


ValueError: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].

In [ ]:
# ===============================
# SETUP + PREPROCESSING + FUNCTIONS
# (Run this ONCE before Stage 1 or Stage 2)
# ===============================

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
import xgboost as xgb

# ====== CONFIG ======
DATA_PATH = "/content/drive/MyDrive/PhD_FraudDetection/data/defi/elliptic_merged.csv"
RESULTS_DIR = "/content/drive/MyDrive/PhD_FraudDetection/results_defi"
os.makedirs(RESULTS_DIR, exist_ok=True)

seed_list = [1011, 2022, 3033, 4044, 5055]
num_random_runs = 5
num_clients = 5

rf_params = {"n_estimators": 150, "n_jobs": -1}
xgb_params = {"eval_metric": "logloss", "verbosity": 0, "n_estimators": 150}
lr_params = {"max_iter": 1000}

# ====== METRICS (supports multiclass) ======
def compute_metrics(y_true, y_pred, y_proba):
    y_true = np.asarray(y_true)
    unique_classes = np.unique(y_true)
    n_classes = len(unique_classes)

    acc = accuracy_score(y_true, y_pred)

    if n_classes <= 2:
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        try:
            auc = roc_auc_score(y_true, y_proba)
        except:
            auc = np.nan
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        return {"Accuracy": acc, "Precision": prec, "Recall": rec,
                "F1": f1, "AUC": auc, "TP": tp, "TN": tn, "FP": fp, "FN": fn}

    else:
        # MULTICLASS
        prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
        rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)
        f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
        try:
            auc = roc_auc_score(y_true, y_proba, multi_class="ovr")
        except:
            auc = np.nan

        cm = confusion_matrix(y_true, y_pred)
        tp = np.diag(cm).sum()
        fp = (cm.sum(axis=0) - np.diag(cm)).sum()
        fn = (cm.sum(axis=1) - np.diag(cm)).sum()
        tn = cm.sum() - (tp + fp + fn)

        return {"Accuracy": acc, "Precision": prec, "Recall": rec,
                "F1": f1, "AUC": auc, "TP": int(tp), "TN": int(tn),
                "FP": int(fp), "FN": int(fn)}

def stratified_client_split(X, y, n_clients=5, random_state=None):
    skf = StratifiedKFold(n_splits=n_clients, shuffle=True, random_state=random_state)
    clients = [(X[idx], y.iloc[idx]) for _, idx in skf.split(X, y)]
    return clients

def average_lr_weights(weights_list):
    coefs = np.array([w[0] for w in weights_list])
    inters = np.array([w[1] for w in weights_list])
    return coefs.mean(axis=0), inters.mean(axis=0)

# ====== LOAD DATA ======
df = pd.read_csv(DATA_PATH)
label_col = "class"

# drop unwanted feature columns
drop_cols = ["txId","Time step","time","timestamp","transactionId","step","node"]
X = df.drop(columns=[label_col] + [c for c in drop_cols if c in df.columns], errors='ignore')
y = df[label_col].astype(int)

# drop non-numeric
non_numeric = X.select_dtypes(include=['object']).columns.tolist()
if non_numeric:
    X = X.drop(columns=non_numeric)

# impute NaNs
imputer = SimpleImputer(strategy="mean")
X_values = imputer.fit_transform(X.values)

# scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_values)

# model factory
def make_models(random_state):
    return {
        "LR": LogisticRegression(random_state=random_state, **lr_params),
        "RF": RandomForestClassifier(random_state=random_state, **rf_params),
        "XGB": xgb.XGBClassifier(random_state=random_state, **xgb_params),
    }

print("Setup completed. Ready for seed-based or random-run execution.")


Setup completed. Ready for seed-based or random-run execution.


In [ ]:
# ===============================
# BLOCK 1 → SEED-BASED EXPERIMENTS
# ===============================

stage1_cent = []
stage1_fl = []

print("\n=========== STAGE 1: SEED-BASED ===========")

for seed in seed_list:

    print(f"\n--- SEED {seed} (CENTRALIZED) ---")
    models = make_models(seed)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    for name, model in models.items():
        model.fit(X_train, y_train)

        y_proba_all = model.predict_proba(X_test)
        y_proba = y_proba_all[:,1] if y_proba_all.shape[1] == 2 else y_proba_all
        y_pred = model.predict(X_test)

        m = compute_metrics(y_test, y_pred, y_proba)
        m.update({"Seed": seed, "Model": name, "Stage": "Centralized"})
        stage1_cent.append(m)
        print(f"[CENT] Seed={seed}, Model={name}, F1={m['F1']:.4f}")

    print(f"\n--- SEED {seed} (FEDERATED) ---")

    clients = stratified_client_split(X_scaled, y, num_clients, seed)
    local_models = {"LR": [], "RF": [], "XGB": []}

    for i, (Xc, yc) in enumerate(clients):
        cm = make_models(seed + i)
        for name, model in cm.items():
            model.fit(Xc, yc)
            local_models[name].append(model)

    for name in ["LR", "RF", "XGB"]:
        if name == "LR":
            weights = [(lm.coef_.copy(), lm.intercept_.copy()) for lm in local_models["LR"]]
            coef, inter = average_lr_weights(weights)
            global_lr = LogisticRegression(**lr_params)
            global_lr.fit(X_train[:5], y_train[:5])
            global_lr.coef_ = coef
            global_lr.intercept_ = inter
            global_lr.classes_ = np.unique(y)
            y_proba_all = global_lr.predict_proba(X_test)
            y_proba = y_proba_all[:,1] if y_proba_all.shape[1] == 2 else y_proba_all
            y_pred = global_lr.predict(X_test)
        else:
            probs = [lm.predict_proba(X_test) for lm in local_models[name]]
            avg_proba = np.mean(np.stack(probs, axis=0), axis=0)
            y_proba = avg_proba[:,1] if avg_proba.shape[1] == 2 else avg_proba
            y_pred = np.argmax(avg_proba, axis=1)

        m = compute_metrics(y_test, y_pred, y_proba)
        m.update({"Seed": seed, "Model": name, "Stage": "Federated"})
        stage1_fl.append(m)
        print(f"[FL] Seed={seed}, Model={name}, F1={m['F1']:.4f}")

# Save
pd.DataFrame(stage1_cent).to_csv(f"{RESULTS_DIR}/stage1_centralized_seed.csv", index=False)
pd.DataFrame(stage1_fl).to_csv(f"{RESULTS_DIR}/stage1_federated_seed.csv", index=False)

print("\nStage 1 Completed.")



=========== STAGE 1: SEED-BASED ===========

--- SEED 1011 (CENTRALIZED) ---
[CENT] Seed=1011, Model=LR, F1=0.8326


KeyboardInterrupt: 

In [ ]:
# ===============================
# RUN ONLY ONE SEED FIRST
# For quick testing
# ===============================

test_seed = 1011  # choose any one seed you want to test
stage1_cent_test = []
stage1_fl_test = []

print(f"\n=========== TESTING SINGLE SEED: {test_seed} ===========")

models = make_models(test_seed)

# Centralized for this seed
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=test_seed
)

print(f"\n--- CENTRALIZED (Seed {test_seed}) ---")
for name, model in models.items():
    model.fit(X_train, y_train)
    y_proba_all = model.predict_proba(X_test)
    y_proba = y_proba_all[:,1] if y_proba_all.shape[1] == 2 else y_proba_all
    y_pred = model.predict(X_test)

    m = compute_metrics(y_test, y_pred, y_proba)
    m.update({"Seed": test_seed, "Model": name, "Stage": "Centralized"})
    stage1_cent_test.append(m)
    print(f"[CENT] Model={name}, F1={m['F1']:.4f}")

# Federated for this seed
print(f"\n--- FEDERATED (Seed {test_seed}) ---")
clients = stratified_client_split(X_scaled, y, num_clients, test_seed)
local_models = {"LR": [], "RF": [], "XGB": []}

# Local training
for i, (Xc, yc) in enumerate(clients):
    cm = make_models(test_seed + i)
    for name, model in cm.items():
        model.fit(Xc, yc)
        local_models[name].append(model)

# Aggregation
for name in ["LR", "RF", "XGB"]:
    if name == "LR":
        weights = [(lm.coef_.copy(), lm.intercept_.copy()) for lm in local_models["LR"]]
        coef, inter = average_lr_weights(weights)

        global_lr = LogisticRegression(**lr_params)
        global_lr.fit(X_train[:5], y_train[:5])
        global_lr.coef_ = coef
        global_lr.intercept_ = inter
        global_lr.classes_ = np.unique(y)
        y_proba_all = global_lr.predict_proba(X_test)
        y_proba = y_proba_all[:,1] if y_proba_all.shape[1] == 2 else y_proba_all
        y_pred = global_lr.predict(X_test)
    else:
        probs = [lm.predict_proba(X_test) for lm in local_models[name]]
        avg_proba = np.mean(np.stack(probs, axis=0), axis=0)
        y_proba = avg_proba[:,1] if avg_proba.shape[1] == 2 else avg_proba
        y_pred = np.argmax(avg_proba, axis=1)

    m = compute_metrics(y_test, y_pred, y_proba)
    m.update({"Seed": test_seed, "Model": name, "Stage": "Federated"})
    stage1_fl_test.append(m)
    print(f"[FL] Model={name}, F1={m['F1']:.4f}")

print("\nSingle-seed test completed.")



=========== TESTING SINGLE SEED: 1011 ===========

--- CENTRALIZED (Seed 1011) ---
[CENT] Model=LR, F1=0.8326


KeyboardInterrupt: 

In [ ]:
# ===============================
# SETUP BLOCK — RUN ONCE
# ===============================
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
import xgboost as xgb

DATA_PATH = "/content/drive/MyDrive/PhD_FraudDetection/data/defi/elliptic_merged.csv"
RESULTS_DIR = "/content/drive/MyDrive/PhD_FraudDetection/results_defi"
os.makedirs(RESULTS_DIR, exist_ok=True)

seed_list = [1011, 2022, 3033, 4044, 5055]

rf_params = {"n_estimators": 150, "n_jobs": -1}
xgb_params = {"eval_metric": "logloss", "verbosity": 0, "n_estimators": 150}
lr_params = {"max_iter": 1000}

# ========== Metric function for multiclass ==========
def compute_metrics(y_true, y_pred, y_proba):
    y_true = np.asarray(y_true)
    unique_classes = np.unique(y_true)
    n_classes = len(unique_classes)

    acc = accuracy_score(y_true, y_pred)

    if n_classes <= 2:
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        try:
            auc = roc_auc_score(y_true, y_proba)
        except:
            auc = np.nan
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1,
                "AUC": auc, "TP": tp, "TN": tn, "FP": fp, "FN": fn}

    else:
        prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
        rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)
        f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
        try:
            auc = roc_auc_score(y_true, y_proba, multi_class="ovr")
        except:
            auc = np.nan

        cm = confusion_matrix(y_true, y_pred)
        tp = np.diag(cm).sum()
        fp = (cm.sum(axis=0) - np.diag(cm)).sum()
        fn = (cm.sum(axis=1) - np.diag(cm)).sum()
        tn = cm.sum() - (tp + fp + fn)

        return {"Accuracy": acc, "Precision": prec, "Recall": rec,
                "F1": f1, "AUC": auc, "TP": int(tp), "TN": int(tn),
                "FP": int(fp), "FN": int(fn)}

# ========== Load and preprocess ==========
df = pd.read_csv(DATA_PATH)
label_col = "class"

drop_cols = ["txId","Time step","time","timestamp","transactionId","step","node"]
X = df.drop(columns=[label_col] + [c for c in drop_cols if c in df.columns], errors="ignore")
y = df[label_col].astype(int)

non_numeric = X.select_dtypes(include=["object"]).columns.tolist()
if non_numeric:
    X = X.drop(columns=non_numeric)

imputer = SimpleImputer(strategy="mean")
X_values = imputer.fit_transform(X.values)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_values)

def make_models(seed):
    return {
        "LR": LogisticRegression(random_state=seed, **lr_params),
        "RF": RandomForestClassifier(random_state=seed, **rf_params),
        "XGB": xgb.XGBClassifier(random_state=seed, **xgb_params),
    }

print("Setup complete. Ready to run centralized-only execution.")


Setup complete. Ready to run centralized-only execution.


In [ ]:
# ===============================
# CENTRALIZED-ONLY (FULL SEED LIST)
# ===============================

stage1_centralized = []

print("\n=========== CENTRALIZED ONLY — ALL SEEDS ===========")

for seed in seed_list:
    print(f"\n--- SEED {seed} ---")
    models = make_models(seed)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    for name, model in models.items():
        model.fit(X_train, y_train)

        proba_all = model.predict_proba(X_test)
        proba = proba_all[:,1] if proba_all.shape[1] == 2 else proba_all
        pred = model.predict(X_test)

        m = compute_metrics(y_test, pred, proba)
        m.update({"Seed": seed, "Model": name, "Stage": "Centralized"})
        stage1_centralized.append(m)

        print(f"[CENT] Seed={seed}, Model={name}, F1={m['F1']:.4f}")

# Save results
pd.DataFrame(stage1_centralized).to_csv(
    f"{RESULTS_DIR}/seed_based_centralized_only.csv", index=False
)

print("\nCentralized seed-based execution completed.")



=========== CENTRALIZED ONLY — ALL SEEDS ===========

--- SEED 1011 ---
[CENT] Seed=1011, Model=LR, F1=0.8326


KeyboardInterrupt: 

In [ ]:
# ===============================
# SETUP BLOCK (FAST EXECUTION VERSION)
# Run this ONCE
# ===============================
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
import xgboost as xgb

DATA_PATH = "/content/drive/MyDrive/PhD_FraudDetection/data/defi/elliptic_merged.csv"
RESULTS_DIR = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_fast"
os.makedirs(RESULTS_DIR, exist_ok=True)

seed_list = [1011, 2022, 3033, 4044, 5055]

# FAST model params
rf_params_fast = {"n_estimators": 60, "n_jobs": -1, "max_depth": 8}
xgb_params_fast = {"eval_metric": "mlogloss", "verbosity": 0, "n_estimators": 60, "max_depth": 6}
lr_params = {"max_iter": 800}

# Metric function (supports multiclass)
def compute_metrics(y_true, y_pred, y_proba):
    y_true = np.asarray(y_true)
    n_classes = len(np.unique(y_true))

    acc = accuracy_score(y_true, y_pred)

    if n_classes <= 2:
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        try:
            auc = roc_auc_score(y_true, y_proba)
        except:
            auc = np.nan
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        return {"Accuracy": acc, "Precision": prec, "Recall": rec,
                "F1": f1, "AUC": auc, "TP": tp, "TN": tn, "FP": fp, "FN": fn}

    else:
        prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        try:
            auc = roc_auc_score(y_true, y_proba, multi_class="ovr")
        except:
            auc = np.nan

        cm = confusion_matrix(y_true, y_pred)
        tp = np.diag(cm).sum()
        fp = (cm.sum(axis=0) - np.diag(cm)).sum()
        fn = (cm.sum(axis=1) - np.diag(cm)).sum()
        tn = cm.sum() - (tp + fp + fn)

        return {"Accuracy": acc, "Precision": prec, "Recall": rec,
                "F1": f1, "AUC": auc, "TP": int(tp), "TN": int(tn),
                "FP": int(fp), "FN": int(fn)}

# Load data
df = pd.read_csv(DATA_PATH)
label_col = "class"

# Drop unused
drop_cols = ["txId","Time step","time","timestamp","transactionId","step","node"]
X = df.drop(columns=[label_col] + [c for c in drop_cols if c in df.columns], errors="ignore")
y = df[label_col].astype(int)

# Fix non-numeric
non_numeric = X.select_dtypes(include=['object']).columns.tolist()
if non_numeric:
    X = X.drop(columns=non_numeric)

# Impute NaN
imputer = SimpleImputer(strategy="mean")
X_values = imputer.fit_transform(X.values)

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_values)

# Fast model generator
def make_fast_models(seed):
    return {
        "LR": LogisticRegression(random_state=seed, **lr_params),
        "RF": RandomForestClassifier(random_state=seed, **rf_params_fast),
        "XGB": xgb.XGBClassifier(random_state=seed, **xgb_params_fast),
    }

print("FAST SETUP COMPLETE — Ready to run centralized FAST seeds.")


FAST SETUP COMPLETE — Ready to run centralized FAST seeds.


In [ ]:
# ===============================
# FAST CENTRALIZED (ALL SEEDS)
# ===============================

stage1_fast = []

print("\n=========== FAST CENTRALIZED (ALL 5 SEEDS) ===========")

for seed in seed_list:
    print(f"\n--- SEED {seed} ---")
    models = make_fast_models(seed)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    for name, model in models.items():
        model.fit(X_train, y_train)

        proba_all = model.predict_proba(X_test)
        proba = proba_all[:,1] if proba_all.shape[1] == 2 else proba_all
        pred = model.predict(X_test)

        m = compute_metrics(y_test, pred, proba)
        m.update({"Seed": seed, "Model": name, "Stage": "Centralized_FAST"})
        stage1_fast.append(m)

        print(f"[FAST-CENT] Seed={seed}, Model={name}, F1={m['F1']:.4f}")

pd.DataFrame(stage1_fast).to_csv(f"{RESULTS_DIR}/fast_centralized_seed_results.csv", index=False)

print("\nFAST CENTRALIZED RUNTIME — Completed Successfully!")



=========== FAST CENTRALIZED (ALL 5 SEEDS) ===========

--- SEED 1011 ---
[FAST-CENT] Seed=1011, Model=LR, F1=0.8326
[FAST-CENT] Seed=1011, Model=RF, F1=0.9016


ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2], got [1 2 3]

In [ ]:
# ===============================
# SETUP BLOCK — RUN THIS FIRST ONCE
# ===============================
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
import xgboost as xgb

# paths
DATA_PATH = "/content/drive/MyDrive/PhD_FraudDetection/data/defi/elliptic_merged.csv"
RESULTS_DIR = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model"
os.makedirs(RESULTS_DIR, exist_ok=True)

seed_list = [1011, 2022, 3033, 4044, 5055]

# FAST model params (optional)
rf_params = {"n_estimators": 100, "n_jobs": -1}
xgb_params = {"eval_metric": "mlogloss", "verbosity": 0, "n_estimators": 100}

# ======== Metric function (supports multiclass) ========
def compute_metrics(y_true, y_pred, y_proba):
    y_true = np.asarray(y_true)
    n_classes = len(np.unique(y_true))

    acc = accuracy_score(y_true, y_pred)

    if n_classes <= 2:
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        try:
            auc = roc_auc_score(y_true, y_proba)
        except:
            auc = np.nan
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        return {"Accuracy": acc, "Precision": prec, "Recall": rec,
                "F1": f1, "AUC": auc, "TP": tp, "TN": tn, "FP": fp, "FN": fn}
    else:
        prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

        try:
            auc = roc_auc_score(y_true, y_proba, multi_class="ovr")
        except:
            auc = np.nan

        cm = confusion_matrix(y_true, y_pred)
        tp = np.diag(cm).sum()
        fp = (cm.sum(axis=0) - np.diag(cm)).sum()
        fn = (cm.sum(axis=1) - np.diag(cm)).sum()
        tn = cm.sum() - (tp + fp + fn)

        return {"Accuracy": acc, "Precision": prec, "Recall": rec,
                "F1": f1, "AUC": auc, "TP": int(tp), "TN": int(tn),
                "FP": int(fp), "FN": int(fn)}

# ========= Load + Preprocess =========
df = pd.read_csv(DATA_PATH)
label_col = "class"

# FIX CLASS LABELS 1,2,3 → 0,1,2
df[label_col] = df[label_col].astype(int) - 1

drop_cols = ["txId","Time step","time","timestamp","transactionId","step","node"]
X = df.drop(columns=[label_col] + [c for c in drop_cols if c in df.columns], errors="ignore")
y = df[label_col].astype(int)

non_numeric = X.select_dtypes(include=['object']).columns.tolist()
if non_numeric:
    X = X.drop(columns=non_numeric)

imputer = SimpleImputer(strategy="mean")
X_values = imputer.fit_transform(X.values)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_values)

print("Setup complete. Ready for model-by-model execution.")


Setup complete. Ready for model-by-model execution.


In [ ]:
# ============================
# MODEL 1 — LOGISTIC REGRESSION (ALL SEEDS)
# ============================

lr_results = []

print("\n========== LOGISTIC REGRESSION (CENTRALIZED) ==========")

for seed in seed_list:
    print(f"\nRunning LR for Seed {seed}...")

    model = LogisticRegression(random_state=seed, max_iter=1000)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    model.fit(X_train, y_train)

    proba_all = model.predict_proba(X_test)
    proba = proba_all[:,1] if proba_all.shape[1] == 2 else proba_all
    pred = model.predict(X_test)

    m = compute_metrics(y_test, pred, proba)
    m.update({"Model": "LR", "Seed": seed})
    lr_results.append(m)

    print(f"[LR] Seed={seed}, F1={m['F1']:.4f}")

pd.DataFrame(lr_results).to_csv(f"{RESULTS_DIR}/LR_seed_results.csv", index=False)

print("\nLR completed for all seeds.")



========== LOGISTIC REGRESSION (CENTRALIZED) ==========

Running LR for Seed 1011...
[LR] Seed=1011, F1=0.8326

Running LR for Seed 2022...
[LR] Seed=2022, F1=0.8333

Running LR for Seed 3033...


KeyboardInterrupt: 

In [ ]:
# ===============================
# METRIC FUNCTION (ONLY MAIN METRICS)
# ===============================
def compute_metrics(y_true, y_pred, y_proba):
    y_true = np.asarray(y_true)
    n_classes = len(np.unique(y_true))

    acc  = accuracy_score(y_true, y_pred)

    if n_classes <= 2:
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec  = recall_score(y_true, y_pred, zero_division=0)
        f1   = f1_score(y_true, y_pred, zero_division=0)
        try:
            auc = roc_auc_score(y_true, y_proba)
        except:
            auc = np.nan
        return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1, "AUC": auc}

    else:
        prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        try:
            auc = roc_auc_score(y_true, y_proba, multi_class="ovr")
        except:
            auc = np.nan

        return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1, "AUC": auc}


In [ ]:
# ============================
# LOGISTIC REGRESSION (SEED-BASED)
# ============================

lr_results = []

print("\n========== LOGISTIC REGRESSION (CENTRALIZED) ==========")

for seed in seed_list:
    print(f"\nRunning LR for Seed {seed}...")

    model = LogisticRegression(random_state=seed, max_iter=1200)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    model.fit(X_train, y_train)

    proba_all = model.predict_proba(X_test)
    proba = proba_all[:,1] if proba_all.shape[1] == 2 else proba_all
    pred = model.predict(X_test)

    m = compute_metrics(y_test, pred, proba)
    m.update({"Model": "LR", "Seed": seed})
    lr_results.append(m)

    print(f"[LR] Seed={seed} | Acc={m['Accuracy']:.4f} | F1={m['F1']:.4f} | AUC={m['AUC']:.4f}")

pd.DataFrame(lr_results).to_csv(f"{RESULTS_DIR}/LR_seed_results.csv", index=False)
print("\nLR Completed.")



========== LOGISTIC REGRESSION (CENTRALIZED) ==========

Running LR for Seed 1011...
[LR] Seed=1011 | Acc=0.8530 | F1=0.8326 | AUC=0.8890

Running LR for Seed 2022...
[LR] Seed=2022 | Acc=0.8537 | F1=0.8333 | AUC=0.8900

Running LR for Seed 3033...
[LR] Seed=3033 | Acc=0.8568 | F1=0.8368 | AUC=0.8914

Running LR for Seed 4044...


KeyboardInterrupt: 

In [ ]:
# ===============================
# METRIC FUNCTION (4 METRICS ONLY)
# ===============================
def compute_metrics(y_true, y_pred, y_proba):
    y_true = np.asarray(y_true)
    n_classes = len(np.unique(y_true))

    acc = accuracy_score(y_true, y_pred)

    # Binary or multiclass both handled
    if n_classes <= 2:
        prec = precision_score(y_true, y_pred, zero_division=0)
        f1   = f1_score(y_true, y_pred, zero_division=0)
        try:
            auc = roc_auc_score(y_true, y_proba)
        except:
            auc = np.nan
        return {"Accuracy": acc, "Precision": prec, "F1": f1, "AUC": auc}

    else:
        prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
        f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)
        try:
            auc = roc_auc_score(y_true, y_proba, multi_class="ovr")
        except:
            auc = np.nan

        return {"Accuracy": acc, "Precision": prec, "F1": f1, "AUC": auc}


In [ ]:
# ============================
# LOGISTIC REGRESSION (ALL SEEDS)
# ============================

lr_results = []

print("\n========== LOGISTIC REGRESSION (4 METRICS) ==========")

for seed in seed_list:
    print(f"\nRunning LR for Seed {seed}...")

    model = LogisticRegression(random_state=seed, max_iter=1200)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    model.fit(X_train, y_train)

    proba_all = model.predict_proba(X_test)
    proba = proba_all[:,1] if proba_all.shape[1] == 2 else proba_all
    pred = model.predict(X_test)

    m = compute_metrics(y_test, pred, proba)
    m.update({"Model": "LR", "Seed": seed})
    lr_results.append(m)

    print(f"[LR] Seed={seed} | Acc={m['Accuracy']:.4f} | Prec={m['Precision']:.4f} | F1={m['F1']:.4f} | AUC={m['AUC']:.4f}")

pd.DataFrame(lr_results).to_csv(f"{RESULTS_DIR}/LR_seed_results.csv", index=False)



========== LOGISTIC REGRESSION (4 METRICS) ==========

Running LR for Seed 1011...
[LR] Seed=1011 | Acc=0.8530 | Prec=0.8313 | F1=0.8326 | AUC=0.8890

Running LR for Seed 2022...
[LR] Seed=2022 | Acc=0.8537 | Prec=0.8383 | F1=0.8333 | AUC=0.8900

Running LR for Seed 3033...
[LR] Seed=3033 | Acc=0.8568 | Prec=0.8376 | F1=0.8368 | AUC=0.8914

Running LR for Seed 4044...
[LR] Seed=4044 | Acc=0.8493 | Prec=0.8307 | F1=0.8281 | AUC=0.8846

Running LR for Seed 5055...
[LR] Seed=5055 | Acc=0.8547 | Prec=0.8370 | F1=0.8346 | AUC=0.8887


In [ ]:
# ============================
# RANDOM FOREST (ALL SEEDS)
# ============================

rf_results = []

print("\n========== RANDOM FOREST (4 METRICS) ==========")

for seed in seed_list:
    print(f"\nRunning RF for Seed {seed}...")

    model = RandomForestClassifier(
        random_state=seed,
        n_estimators=150,
        n_jobs=-1
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    model.fit(X_train, y_train)

    proba_all = model.predict_proba(X_test)
    proba = proba_all[:,1] if proba_all.shape[1] == 2 else proba_all
    pred = model.predict(X_test)

    m = compute_metrics(y_test, pred, proba)
    m.update({"Model": "RF", "Seed": seed})
    rf_results.append(m)

    print(f"[RF] Seed={seed} | Acc={m['Accuracy']:.4f} | Prec={m['Precision']:.4f} | F1={m['F1']:.4f} | AUC={m['AUC']:.4f}")

pd.DataFrame(rf_results).to_csv(f"{RESULTS_DIR}/RF_seed_results.csv", index=False)



========== RANDOM FOREST (4 METRICS) ==========

Running RF for Seed 1011...
[RF] Seed=1011 | Acc=0.9516 | Prec=0.9512 | F1=0.9505 | AUC=0.9889

Running RF for Seed 2022...
[RF] Seed=2022 | Acc=0.9542 | Prec=0.9539 | F1=0.9531 | AUC=0.9897

Running RF for Seed 3033...
[RF] Seed=3033 | Acc=0.9522 | Prec=0.9518 | F1=0.9511 | AUC=0.9885

Running RF for Seed 4044...
[RF] Seed=4044 | Acc=0.9512 | Prec=0.9508 | F1=0.9500 | AUC=0.9882

Running RF for Seed 5055...
[RF] Seed=5055 | Acc=0.9528 | Prec=0.9525 | F1=0.9517 | AUC=0.9892


In [ ]:
# ============================
# XGBOOST (ALL SEEDS)
# ============================

xgb_results = []

print("\n========== XGBOOST (4 METRICS) ==========")

for seed in seed_list:
    print(f"\nRunning XGB for Seed {seed}...")

    model = xgb.XGBClassifier(
        random_state=seed,
        eval_metric="mlogloss",
        verbosity=0,
        n_estimators=150
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    model.fit(X_train, y_train)

    proba_all = model.predict_proba(X_test)
    proba = proba_all[:,1] if proba_all.shape[1] == 2 else proba_all
    pred = model.predict(X_test)

    m = compute_metrics(y_test, pred, proba)
    m.update({"Model": "XGB", "Seed": seed})
    xgb_results.append(m)

    print(f"[XGB] Seed={seed} | Acc={m['Accuracy']:.4f} | Prec={m['Precision']:.4f} | F1={m['F1']:.4f} | AUC={m['AUC']:.4f}")

pd.DataFrame(xgb_results).to_csv(f"{RESULTS_DIR}/XGB_seed_results.csv", index=False)



========== XGBOOST (4 METRICS) ==========

Running XGB for Seed 1011...
[XGB] Seed=1011 | Acc=0.9594 | Prec=0.9589 | F1=0.9589 | AUC=0.9925

Running XGB for Seed 2022...
[XGB] Seed=2022 | Acc=0.9597 | Prec=0.9592 | F1=0.9592 | AUC=0.9923

Running XGB for Seed 3033...
[XGB] Seed=3033 | Acc=0.9604 | Prec=0.9600 | F1=0.9599 | AUC=0.9917

Running XGB for Seed 4044...
[XGB] Seed=4044 | Acc=0.9586 | Prec=0.9581 | F1=0.9580 | AUC=0.9916

Running XGB for Seed 5055...
[XGB] Seed=5055 | Acc=0.9598 | Prec=0.9594 | F1=0.9594 | AUC=0.9924


In [ ]:
!pip install reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.9 MB/s eta 0:00:00


In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
import pandas as pd
import datetime
import os

# ============================
# CONFIG
# ============================
results_dir = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model"
pdf_path = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/DeFi_Centralized_Seed_Results.pdf"

# File names
lr_file  = f"{results_dir}/LR_seed_results.csv"
rf_file  = f"{results_dir}/RF_seed_results.csv"
xgb_file = f"{results_dir}/XGB_seed_results.csv"

# Load CSV files
df_lr  = pd.read_csv(lr_file)
df_rf  = pd.read_csv(rf_file)
df_xgb = pd.read_csv(xgb_file)

# ============================
# PDF Document Setup
# ============================
doc = SimpleDocTemplate(pdf_path, pagesize=letter)
styles = getSampleStyleSheet()
story = []

# Title
title = Paragraph("<b>DeFi Centralized Model Results (Seed-Based)</b>", styles["Title"])
story.append(title)
story.append(Spacer(1, 12))

# Date
date_text = Paragraph(f"Generated on: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", styles["Normal"])
story.append(date_text)
story.append(Spacer(1, 20))


# ============================
# FUNCTION: Add model section
# ============================
def add_model_section(model_name, df, story):
    story.append(Paragraph(f"<b>{model_name} Results</b>", styles["Heading2"]))
    story.append(Spacer(1, 12))

    # Keep only needed columns
    cols = ["Seed", "Accuracy", "Precision", "F1", "AUC"]
    df = df[cols]

    # Convert dataframe to PDF table format
    table_data = [df.columns.tolist()] + df.values.tolist()

    table = Table(table_data)
    table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.lightgrey),
        ('TEXTCOLOR', (0,0), (-1,0), colors.black),
        ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
        ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ('ALIGN', (1,1), (-1,-1), 'CENTER'),
        ('FONTNAME', (0,1), (-1,-1), 'Helvetica')
    ]))

    story.append(table)
    story.append(Spacer(1, 20))


# ============================
# Add LR, RF, XGB sections
# ============================
add_model_section("Logistic Regression", df_lr, story)
add_model_section("Random Forest", df_rf, story)
add_model_section("XGBoost", df_xgb, story)

# ============================
# Build PDF
# ============================
doc.build(story)

print("PDF generated successfully at:")
print(pdf_path)


PDF generated successfully at:
/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/DeFi_Centralized_Seed_Results.pdf


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


SETUP COMPLETE — X_scaled and y are ready.


Random seeds selected for random-run evaluation:
[88639, 98015, 16339, 11103, 81485]

=========== RANDOM RUN (CENTRALIZED with AUTO-SEEDS) ===========

===== RANDOM RUN (Seed = 88639) =====

Running LR for Random Seed 88639...
[LR] Seed=88639 | Acc=0.8565 | Prec=0.8384 | F1=0.8365 | AUC=0.8859

Running RF for Random Seed 88639...
[RF] Seed=88639 | Acc=0.9514 | Prec=0.9508 | F1=0.9504 | AUC=0.9873

Running XGB for Random Seed 88639...
[XGB] Seed=88639 | Acc=0.9576 | Prec=0.9572 | F1=0.9572 | AUC=0.9909

===== RANDOM RUN (Seed = 98015) =====

Running LR for Random Seed 98015...
[LR] Seed=98015 | Acc=0.8544 | Prec=0.8366 | F1=0.8338 | AUC=0.8882

Running RF for Random Seed 98015...
[RF] Seed=98015 | Acc=0.9512 | Prec=0.9508 | F1=0.9502 | AUC=0.9879

Running XGB for Random Seed 98015...
[XGB] Seed=98015 | Acc=0.9585 | Prec=0.9581 | F1=0.9581 | AUC=0.9917

===== RANDOM RUN (Seed = 16339) =====

Running LR for Random Seed 16339...
[LR] Seed=16339 | Acc=0.8551 | Prec=0.8361 | F1=0.8348 | AUC=

In [ ]:
!pip install reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 14.7 MB/s eta 0:00:00


In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
import pandas as pd
import datetime

# ============================
# FILE PATHS
# ============================
results_dir = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model"

lr_csv  = f"{results_dir}/random_run_LR_results.csv"
rf_csv  = f"{results_dir}/random_run_RF_results.csv"
xgb_csv = f"{results_dir}/random_run_XGB_results.csv"

pdf_path = f"{results_dir}/DeFi_Centralized_RandomRun_Results.pdf"

# Load CSVs
df_lr  = pd.read_csv(lr_csv)
df_rf  = pd.read_csv(rf_csv)
df_xgb = pd.read_csv(xgb_csv)

# ============================
# PDF Setup
# ============================
doc = SimpleDocTemplate(pdf_path, pagesize=letter)
styles = getSampleStyleSheet()
story = []

# Title
title = Paragraph("<b>DeFi Centralized Random Run Results</b>", styles["Title"])
story.append(title)
story.append(Spacer(1, 12))

# Date
date_text = Paragraph(f"Generated On: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
                      styles["Normal"])
story.append(date_text)
story.append(Spacer(1, 20))

# ============================
# Helper Function to Add Tables
# ============================
def add_model_section(model_name, df, story):
    story.append(Paragraph(f"<b>{model_name} — Random Run Results</b>", styles["Heading2"]))
    story.append(Spacer(1, 10))

    # Keep only important columns
    cols = ["Random_Run_Seed", "Accuracy", "Precision", "F1", "AUC"]
    df = df[cols]

    # Convert to table format
    table_data = [df.columns.tolist()] + df.values.tolist()

    table = Table(table_data)
    table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.lightgrey),
        ('TEXTCOLOR', (0,0), (-1,0), colors.black),
        ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
        ('ALIGN', (1,1), (-1,-1), 'CENTER'),
        ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ('FONTNAME', (0,1), (-1,-1), 'Helvetica')
    ]))

    story.append(table)
    story.append(Spacer(1, 20))

# ============================
# Add three model sections
# ============================
add_model_section("Logistic Regression (LR)", df_lr, story)
add_model_section("Random Forest (RF)", df_rf, story)
add_model_section("XGBoost (XGB)", df_xgb, story)

# ============================
# Build PDF
# ============================
doc.build(story)

print("PDF generated successfully at:")
print(pdf_path)


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/random_run_LR_results.csv'

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
import pandas as pd
import datetime

# Path
csv_path = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/random_run_centralized_results.csv"
pdf_path = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/DeFi_Centralized_RandomRun_Results.pdf"

# Load combined CSV
df = pd.read_csv(csv_path)

# PDF setup
doc = SimpleDocTemplate(pdf_path, pagesize=letter)
styles = getSampleStyleSheet()
story = []

# Title
title = Paragraph("<b>DeFi Centralized Random Run Results</b>", styles["Title"])
story.append(title)
story.append(Spacer(1, 12))

# Date
date_text = Paragraph(f"Generated On: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
                      styles["Normal"])
story.append(date_text)
story.append(Spacer(1, 20))


# ============================
# Helper to add model tables
# ============================
def add_table(model_name):
    story.append(Paragraph(f"<b>{model_name} Results</b>", styles["Heading2"]))
    story.append(Spacer(1, 12))

    temp = df[df["Model"] == model_name][
        ["Random_Run", "Accuracy", "Precision", "F1", "AUC"]
    ]

    table_data = [temp.columns.tolist()] + temp.values.tolist()

    table = Table(table_data)
    table_style = TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.lightgrey),
        ('TEXTCOLOR', (0,0), (-1,0), colors.black),
        ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
        ('ALIGN', (1,1), (-1,-1), 'CENTER'),
        ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ])
    table.setStyle(table_style)

    story.append(table)
    story.append(Spacer(1, 20))


# Add LR, RF, XGB sections
add_table("LR")
add_table("RF")
add_table("XGB")

# Build PDF
doc.build(story)

print("PDF generated successfully at:")
print(pdf_path)


KeyError: "['Random_Run'] not in index"

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/random_run_centralized_results.csv")
print(df.columns)
df.head()


Index(['Accuracy', 'Precision', 'F1', 'AUC', 'Model', 'Random_Run_Seed'], dtype='object')


,Accuracy,Precision,F1,AUC,Model,Random_Run_Seed
0,0.856505,0.838404,0.836454,0.885943,LR,88639
1,0.951391,0.950810,0.950401,0.987270,RF,88639
2,0.957648,0.957157,0.957216,0.990922,XGB,88639
3,0.854395,0.836627,0.833827,0.888162,LR,98015
4,0.951244,0.950784,0.950235,0.987898,RF,98015


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/random_run_centralized_results.csv")
print(df.columns)
df.head()


Index(['Accuracy', 'Precision', 'F1', 'AUC', 'Model', 'Random_Run_Seed'], dtype='object')


,Accuracy,Precision,F1,AUC,Model,Random_Run_Seed
0,0.856505,0.838404,0.836454,0.885943,LR,88639
1,0.951391,0.950810,0.950401,0.987270,RF,88639
2,0.957648,0.957157,0.957216,0.990922,XGB,88639
3,0.854395,0.836627,0.833827,0.888162,LR,98015
4,0.951244,0.950784,0.950235,0.987898,RF,98015


In [ ]:
!pip install reportlab


In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
import pandas as pd
import datetime

# ============================
# Paths
# ============================
csv_path = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/random_run_centralized_results.csv"
pdf_path = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/DeFi_Centralized_RandomRun_Results.pdf"

# Load CSV
df = pd.read_csv(csv_path)

# Use the correct seed column
seed_col = "Random_Run_Seed"

# ============================
# PDF setup
# ============================
doc = SimpleDocTemplate(pdf_path, pagesize=letter)
styles = getSampleStyleSheet()
story = []

# ============================
# Title page
# ============================
title = Paragraph("<b>DeFi Centralized — Random Run Results (Elliptic Dataset)</b>", styles["Title"])
story.append(title)
story.append(Spacer(1, 14))

date_text = Paragraph(
    f"Generated On: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    styles["Normal"]
)
story.append(date_text)
story.append(Spacer(1, 20))


# ============================
# Function to add model table
# ============================
def add_model_section(model_name):
    story.append(Paragraph(f"<b>{model_name} — Random Run Results</b>", styles["Heading2"]))
    story.append(Spacer(1, 10))

    temp = df[df["Model"] == model_name][[seed_col, "Accuracy", "Precision", "F1", "AUC"]]

    table_data = [temp.columns.tolist()] + temp.values.tolist()

    table = Table(table_data)
    table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.lightgrey),
        ('TEXTCOLOR', (0,0), (-1,0), colors.black),
        ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
        ('ALIGN', (1,1), (-1,-1), 'CENTER'),
        ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ('FONTNAME', (0,1), (-1,-1), 'Helvetica'),
    ]))

    story.append(table)
    story.append(Spacer(1, 20))


# ============================
# Add sections for LR, RF, XGB
# ============================
add_model_section("LR")
add_model_section("RF")
add_model_section("XGB")

# ============================
# Build PDF
# ============================
doc.build(story)

print("PDF generated successfully at:")
print(pdf_path)


PDF generated successfully at:
/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/DeFi_Centralized_RandomRun_Results.pdf


In [ ]:
# ==========================================
# CENTRALIZED – SEED-BASED RESULTS (DEFI)
# Seeds: [1011, 2022, 3033, 4044, 5055]
# ==========================================

seed_list = [1011, 2022, 3033, 4044, 5055]

seed_results = []

print("\n======= CENTRALIZED SEED-BASED RESULTS (ELLIPTIC DATASET) =======\n")

models = {
    "LR":  LogisticRegression(max_iter=1200),
    "RF":  RandomForestClassifier(n_estimators=150, n_jobs=-1),
    "XGB": xgb.XGBClassifier(eval_metric="mlogloss", verbosity=0, n_estimators=150)
}

for seed in seed_list:
    print(f"\n----- SEED {seed} -----")

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    for name, base_model in models.items():

        model = base_model.set_params(random_state=seed)
        model.fit(X_train, y_train)

        proba_all = model.predict_proba(X_test)
        proba = proba_all[:,1] if proba_all.shape[1] == 2 else proba_all
        pred = model.predict(X_test)

        m = compute_metrics(y_test, pred, proba)
        m.update({"Model": name, "Seed": seed})
        seed_results.append(m)

        print(f"{name}:  Acc={m['Accuracy']:.4f} | Prec={m['Precision']:.4f} | "
              f"F1={m['F1']:.4f} | AUC={m['AUC']:.4f}")

# Save to CSV
df_seed = pd.DataFrame(seed_results)
df_seed.to_csv(
    "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/seed_based_centralized_results.csv",
    index=False
)

print("\nSaved seed-based results to:")
print("/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/seed_based_centralized_results.csv")



======= CENTRALIZED SEED-BASED RESULTS (ELLIPTIC DATASET) =======


----- SEED 1011 -----
LR:  Acc=0.8530 | Prec=0.8313 | F1=0.8326 | AUC=0.8890


KeyboardInterrupt: 

In [ ]:
#DEFI FL PART


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
X_scaled and y are ready.


SyntaxError: invalid syntax (ipython-input-3245874048.py, line 1)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Load dataset
path = "/content/drive/MyDrive/PhD_FraudDetection/data/defi/elliptic_merged.csv"
df = pd.read_csv(path)

print("Loaded:", path)
print("Shape:", df.shape)

# Detect label column
label_col = "class"
y = df[label_col].values

# Drop non-feature columns
X = df.drop(columns=[label_col, "txId", "Time step"], errors="ignore")

# Impute missing values
imp = SimpleImputer(strategy="mean")
X_imputed = imp.fit_transform(X)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print("X_scaled shape:", X_scaled.shape)
print("y shape:", y.shape)
print("Dataset ready.")


Loaded: /content/drive/MyDrive/PhD_FraudDetection/data/defi/elliptic_merged.csv
Shape: (203769, 185)
X_scaled shape: (203769, 182)
y shape: (203769,)
Dataset ready.


In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    from sklearn.metrics import accuracy_score, precision_score, f1_score, roc_auc_score
    y_true = np.asarray(y_true)
    n_classes = len(np.unique(y_true))

    acc = accuracy_score(y_true, y_pred)

    if n_classes <= 2:
        prec = precision_score(y_true, y_pred, zero_division=0)
        f1   = f1_score(y_true, y_pred, zero_division=0)
        try:
            auc = roc_auc_score(y_true, y_proba)
        except:
            auc = np.nan
        return {"Accuracy": acc, "Precision": prec, "F1": f1, "AUC": auc}

    else:
        prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        try:
            auc = roc_auc_score(y_true, y_proba, multi_class="ovr")
        except:
            auc = np.nan

        return {"Accuracy": acc, "Precision": prec, "F1": f1, "AUC": auc}

print("Metric function ready.")


Metric function ready.


In [ ]:
# ============================
# BLOCK A — FL SETUP (Run once)
# ============================

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# Paths
RESULTS_DIR = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model"
os.makedirs(RESULTS_DIR, exist_ok=True)

OUT_CSV = f"{RESULTS_DIR}/federated_seed_based_results.csv"

# Number of clients
NUM_CLIENTS = 5

# Seed list for seed-based FL
seed_list = [1011, 2022, 3033, 4044, 5055]

# ========== stratified client split ==========
def stratified_client_split(X, y, n_clients=5, random_state=None):
    """
    Returns a list of (X_client, y_client) for each client.
    Splits data into n stratified partitions.
    """
    skf = StratifiedKFold(n_splits=n_clients, shuffle=True, random_state=random_state)
    clients = []
    X_np = np.asarray(X)
    y_np = np.asarray(y)

    for _, idx in skf.split(X_np, y_np):
        clients.append((X_np[idx], y_np[idx]))

    return clients

print("FL setup complete.")
print("NUM_CLIENTS =", NUM_CLIENTS)
print("Seeds =", seed_list)


FL setup complete.
NUM_CLIENTS = 5
Seeds = [1011, 2022, 3033, 4044, 5055]


In [ ]:
# ============================
# BLOCK B — Federated Logistic Regression (FedAvg) — Seed-based
# ============================
from copy import deepcopy

results = []  # collect results for CSV

print("\n=== Federated LR (FedAvg) — Seed-based ===")

for seed in seed_list:
    print(f"\n--- Seed {seed} ---")

    # Create stratified client splits
    clients = stratified_client_split(X_scaled, y, n_clients=NUM_CLIENTS, random_state=seed)

    # Create global test split (same as centralized)
    X_train_glob, X_test_glob, y_train_glob, y_test_glob = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    # Train local LR models
    local_weights = []
    classes_ = None

    for i, (Xc, yc) in enumerate(clients):
        lm = LogisticRegression(max_iter=1200)
        lm.fit(Xc, yc)
        local_weights.append((lm.coef_.copy(), lm.intercept_.copy()))
        classes_ = lm.classes_

    # ===== FedAvg (average coefficients & intercepts) =====
    coefs = np.array([w[0] for w in local_weights])      # (n_clients, n_classes, n_features)
    inters = np.array([w[1] for w in local_weights])     # (n_clients, n_classes)

    avg_coef = coefs.mean(axis=0)
    avg_inter = inters.mean(axis=0)

    # Construct global LR
    global_lr = LogisticRegression(max_iter=1200)

    # Fit a tiny subset to initialize internal attributes
    try:
        global_lr.fit(X_train_glob[:2], y_train_glob[:2])
    except:
        global_lr.classes_ = np.unique(y)

    global_lr.coef_ = avg_coef
    global_lr.intercept_ = avg_inter
    global_lr.classes_ = classes_ if classes_ is not None else np.unique(y)

    # Evaluate
    y_proba_all = global_lr.predict_proba(X_test_glob)
    y_proba = y_proba_all[:,1] if y_proba_all.shape[1] == 2 else y_proba_all
    y_pred = global_lr.predict(X_test_glob)

    m = compute_metrics(y_test_glob, y_pred, y_proba)
    m.update({"Model": "LR_FedAvg", "Seed": seed})
    results.append(m)

    print(f"[LR_FedAvg] Seed={seed} | Acc={m['Accuracy']:.4f} | Prec={m['Precision']:.4f} "
          f"| F1={m['F1']:.4f} | AUC={m['AUC']:.4f}")

# Save partial results
pd.DataFrame(results).to_csv(OUT_CSV, index=False)

print("\nSaved LR FedAvg results to:")
print(OUT_CSV)



=== Federated LR (FedAvg) — Seed-based ===

--- Seed 1011 ---
[LR_FedAvg] Seed=1011 | Acc=0.8534 | Prec=0.8321 | F1=0.8330 | AUC=0.8836

--- Seed 2022 ---
[LR_FedAvg] Seed=2022 | Acc=0.8549 | Prec=0.8390 | F1=0.8346 | AUC=0.8855

--- Seed 3033 ---
[LR_FedAvg] Seed=3033 | Acc=0.8572 | Prec=0.8392 | F1=0.8373 | AUC=0.8851

--- Seed 4044 ---
[LR_FedAvg] Seed=4044 | Acc=0.8503 | Prec=0.8339 | F1=0.8291 | AUC=0.8793

--- Seed 5055 ---
[LR_FedAvg] Seed=5055 | Acc=0.8560 | Prec=0.8417 | F1=0.8359 | AUC=0.8834

Saved LR FedAvg results to:
/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/federated_seed_based_results.csv


In [ ]:
# ================================================================
# FEDERATED RF (Probability Aggregation)
# DEFI — ELLIPTIC DATASET — SEED-BASED
# ================================================================

rf_results = []

print("\n=== Federated RF (probability aggregation) — Seed-based (ELLIPTIC DATASET) ===")

for seed in seed_list:
    print(f"\n--- Seed {seed} ---")

    # Create client partitions
    clients = stratified_client_split(X_scaled, y, n_clients=NUM_CLIENTS, random_state=seed)

    # Global evaluation split
    X_train_glob, X_test_glob, y_train_glob, y_test_glob = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    # Train RF on each client
    local_models = []
    for i, (Xc, yc) in enumerate(clients):
        rm = RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=seed + i)
        rm.fit(Xc, yc)
        local_models.append(rm)

    # Average probabilities across clients (FedProb)
    probs = [lm.predict_proba(X_test_glob) for lm in local_models]
    avg_proba = np.mean(np.stack(probs, axis=0), axis=0)

    # Convert aggregated probabilities to predictions
    if avg_proba.shape[1] > 2:  # multiclass
        y_pred = np.argmax(avg_proba, axis=1)
        y_proba = avg_proba
    else:                       # binary-ish scenario
        y_pred = (avg_proba[:,1] >= 0.5).astype(int)
        y_proba = avg_proba[:,1]

    # Compute metrics
    m = compute_metrics(y_test_glob, y_pred, y_proba)
    m.update({"Model": "RF_FedProb", "Seed": seed})

    rf_results.append(m)
    results.append(m)

    print(f"[RF_FedProb] Seed={seed} | Acc={m['Accuracy']:.4f} | Prec={m['Precision']:.4f} "
          f"| F1={m['F1']:.4f} | AUC={m['AUC']:.4f}")

# Save to same CSV
pd.DataFrame(results).to_csv(OUT_CSV, index=False)

print("\nRF Federated results added to:")
print(OUT_CSV)



=== Federated RF (probability aggregation) — Seed-based (ELLIPTIC DATASET) ===

--- Seed 1011 ---
[RF_FedProb] Seed=1011 | Acc=0.0316 | Prec=0.0080 | F1=0.0128 | AUC=0.9951

--- Seed 2022 ---
[RF_FedProb] Seed=2022 | Acc=0.0301 | Prec=0.0076 | F1=0.0121 | AUC=0.9954

--- Seed 3033 ---
[RF_FedProb] Seed=3033 | Acc=0.0308 | Prec=0.0078 | F1=0.0125 | AUC=0.9951

--- Seed 4044 ---
[RF_FedProb] Seed=4044 | Acc=0.0328 | Prec=0.0083 | F1=0.0132 | AUC=0.9950

--- Seed 5055 ---
[RF_FedProb] Seed=5055 | Acc=0.0309 | Prec=0.0078 | F1=0.0124 | AUC=0.9953

RF Federated results added to:
/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/federated_seed_based_results.csv


In [ ]:
# ================================================================
# FIXED FEDERATED RF (Probability Aggregation with Class Alignment)
# DEFI — ELLIPTIC DATASET — SEED-BASED
# ================================================================

rf_results = []

print("\n=== Federated RF FIXED (probability aggregation with class alignment) — Seed-based (ELLIPTIC DATASET) ===")

global_classes = np.unique(y)   # Should be [1,2,3]

for seed in seed_list:
    print(f"\n--- Seed {seed} ---")

    # Create stratified clients
    clients = stratified_client_split(X_scaled, y, n_clients=NUM_CLIENTS, random_state=seed)

    # Global evaluation split
    X_train_glob, X_test_glob, y_train_glob, y_test_glob = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    # Train RF on each client
    local_models = []
    for i, (Xc, yc) in enumerate(clients):
        rm = RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=seed + i)
        rm.fit(Xc, yc)
        local_models.append(rm)

    # ===== Corrected Probability Alignment =====
    aligned_probs = []

    for lm in local_models:
        proba = lm.predict_proba(X_test_glob)
        aligned = np.zeros((len(X_test_glob), len(global_classes)))

        # Align local "classes_" to global order
        for idx, cls in enumerate(lm.classes_):
            target_col = np.where(global_classes == cls)[0][0]
            aligned[:, target_col] = proba[:, idx]

        aligned_probs.append(aligned)

    # Average aligned probabilities
    avg_proba = np.mean(np.stack(aligned_probs, axis=0), axis=0)

    # Predictions
    if avg_proba.shape[1] > 2:  # multiclass
        y_pred = np.argmax(avg_proba, axis=1)
        y_proba_for_auc = avg_proba
    else:
        y_pred = (avg_proba[:,1] >= 0.5).astype(int)
        y_proba_for_auc = avg_proba[:,1]

    # Compute metrics
    m = compute_metrics(y_test_glob, y_pred, y_proba_for_auc)
    m.update({"Model": "RF_FedProb_Fixed", "Seed": seed})

    rf_results.append(m)
    results.append(m)

    print(f"[RF_FedProb_FIXED] Seed={seed} | Acc={m['Accuracy']:.4f} | Prec={m['Precision']:.4f} "
          f"| F1={m['F1']:.4f} | AUC={m['AUC']:.4f}")

# Save updated results
pd.DataFrame(results).to_csv(OUT_CSV, index=False)

print("\n✔ FIXED RF Federated results saved to:")
print(OUT_CSV)



=== Federated RF FIXED (probability aggregation with class alignment) — Seed-based (ELLIPTIC DATASET) ===

--- Seed 1011 ---
[RF_FedProb_FIXED] Seed=1011 | Acc=0.0316 | Prec=0.0080 | F1=0.0128 | AUC=0.9951

--- Seed 2022 ---
[RF_FedProb_FIXED] Seed=2022 | Acc=0.0301 | Prec=0.0076 | F1=0.0121 | AUC=0.9954

--- Seed 3033 ---


KeyboardInterrupt: 

In [ ]:
# DEBUG: Inspect RF averaged probabilities for Seed 1011
seed = 1011

clients = stratified_client_split(X_scaled, y, NUM_CLIENTS, random_state=seed)

# Global test split
X_train_glob, X_test_glob, y_train_glob, y_test_glob = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=seed
)

local_models = []
for i, (Xc, yc) in enumerate(clients):
    rm = RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=seed+i)
    rm.fit(Xc, yc)
    local_models.append(rm)

# Align probabilities
global_classes = np.unique(y)
aligned_probs = []

for lm in local_models:
    proba = lm.predict_proba(X_test_glob)
    aligned = np.zeros((len(X_test_glob), len(global_classes)))

    for idx, cls in enumerate(lm.classes_):
        target_col = np.where(global_classes == cls)[0][0]
        aligned[:, target_col] = proba[:, idx]

    aligned_probs.append(aligned)

avg_proba = np.mean(np.stack(aligned_probs, axis=0), axis=0)

print("Avg Proba Sample (first 10 rows):")
print(avg_proba[:10])



In [ ]:
Avg Proba Sample (first 10 rows):
...


SyntaxError: invalid syntax (ipython-input-3296508168.py, line 1)

In [ ]:
print("Avg Proba Sample (first 10 rows):")
print(avg_proba[:10])


Avg Proba Sample (first 10 rows):
[[0.02666667 0.09466667 0.87866667]
 [0.008      0.18133333 0.81066667]
 [0.05333333 0.076      0.87066667]
 [0.00266667 0.86666667 0.13066667]
 [0.01466667 0.24133333 0.744     ]
 [0.00266667 0.01333333 0.984     ]
 [0.008      0.04133333 0.95066667]
 [0.004      0.01466667 0.98133333]
 [0.71866667 0.         0.28133333]
 [0.00133333 0.00533333 0.99333333]]


In [ ]:
# ================================================================
# FIXED FEDERATED RF — Majority Vote Aggregation
# DEFI — ELLIPTIC DATASET — SEED-BASED
# ================================================================

import os
import numpy as np
import pandas as pd
from collections import Counter

print("\n=== Federated RF (Majority Vote aggregation) — Seed-based (ELLIPTIC DATASET) ===")

global_classes = np.unique(y)   # e.g. [0,1,2] or similar

# load existing results if present, so we append without losing LR results
if os.path.exists(OUT_CSV):
    existing_df = pd.read_csv(OUT_CSV)
    # convert to list of dicts for appending
    results = existing_df.to_dict(orient="records")
else:
    results = []


def majority_vote_from_local_preds(local_preds_array, tie_breaker_probas=None):
    """
    local_preds_array: shape (n_clients, n_samples)
    returns array of final predictions (n_samples,)
    tie_breaker_probas: if provided (n_samples, n_classes) used to break ties by argmax of proba
    """
    n_clients, n_samples = local_preds_array.shape
    final_preds = np.zeros(n_samples, dtype=int)

    for j in range(n_samples):
        votes = local_preds_array[:, j]
        # count votes for each class (ensure length matches global_classes)
        counts = np.bincount(votes, minlength=len(global_classes))
        top = np.flatnonzero(counts == counts.max())
        if len(top) == 1:
            final_preds[j] = top[0]
        else:
            # tie: use tie_breaker_probas if available, else pick smallest class index
            if tie_breaker_probas is not None:
                final_preds[j] = np.argmax(tie_breaker_probas[j])
            else:
                final_preds[j] = top[0]
    return final_preds

rf_new_results = []

for seed in seed_list:
    print(f"\n--- Seed {seed} ---")

    # Create client partitions
    clients = stratified_client_split(X_scaled, y, n_clients=NUM_CLIENTS, random_state=seed)

    # Global evaluation split
    X_train_glob, X_test_glob, y_train_glob, y_test_glob = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    # Train RF on each client
    local_models = []
    local_preds = []
    aligned_probs = []

    for i, (Xc, yc) in enumerate(clients):
        rm = RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=seed + i)
        rm.fit(Xc, yc)
        local_models.append(rm)

        # client predictions (for majority vote)
        pred_c = rm.predict(X_test_glob)  # shape (n_samples,)
        local_preds.append(pred_c)

        # align probabilities
        proba = rm.predict_proba(X_test_glob)  # shape (n_samples, n_local_classes)
        aligned = np.zeros((len(X_test_glob), len(global_classes)))
        for idx, cls in enumerate(rm.classes_):
            target_col = np.where(global_classes == cls)[0][0]
            aligned[:, target_col] = proba[:, idx]
        aligned_probs.append(aligned)

    # Stack local_preds to (n_clients, n_samples)
    local_preds_array = np.vstack(local_preds)  # shape (n_clients, n_samples)

    # Majority vote predictions (ties broken by avg_proba argmax)
    avg_proba = np.mean(np.stack(aligned_probs, axis=0), axis=0)  # (n_samples, n_classes)
    vote_preds = majority_vote_from_local_preds(local_preds_array, tie_breaker_probas=avg_proba)

    # As a fallback, where majority_vote equals some invalid class (just in case), use argmax
    invalid_mask = (vote_preds >= len(global_classes))
    if invalid_mask.any():
        vote_preds[invalid_mask] = np.argmax(avg_proba[invalid_mask], axis=1)

    # For metric function, provide appropriate y_proba form:
    if avg_proba.shape[1] > 2:
        y_proba_for_auc = avg_proba
    else:
        y_proba_for_auc = avg_proba[:, 1]  # binary-like

    # Compute metrics using majority-vote predictions
    m = compute_metrics(y_test_glob, vote_preds, y_proba_for_auc)
    m.update({"Model": "RF_Fed_MajorityVote", "Seed": seed})
    rf_new_results.append(m)
    results.append(m)

    print(f"[RF_Fed_MajorityVote] Seed={seed} | Acc={m['Accuracy']:.4f} | Prec={m['Precision']:.4f} "
          f"| F1={m['F1']:.4f} | AUC={m['AUC']:.4f}")

# Save updated results (overwrite OUT_CSV)
pd.DataFrame(results).to_csv(OUT_CSV, index=False)
print("\n✔ FIXED RF Federated (majority vote) results saved to:")
print(OUT_CSV)



=== Federated RF (Majority Vote aggregation) — Seed-based (ELLIPTIC DATASET) ===

--- Seed 1011 ---
[RF_Fed_MajorityVote] Seed=1011 | Acc=0.2170 | Prec=0.0591 | F1=0.0860 | AUC=0.9951

--- Seed 2022 ---
[RF_Fed_MajorityVote] Seed=2022 | Acc=0.2165 | Prec=0.0587 | F1=0.0855 | AUC=0.9954

--- Seed 3033 ---
[RF_Fed_MajorityVote] Seed=3033 | Acc=0.2164 | Prec=0.0585 | F1=0.0855 | AUC=0.9951

--- Seed 4044 ---


KeyboardInterrupt: 

In [ ]:
# ================================================================
# CORRECTED PREPROCESSING BLOCK
# DEFI — ELLIPTIC DATASET
# Fix: Re-encode class labels to 0,1,2 and recompute X_scaled
# ================================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Load dataset
path = "/content/drive/MyDrive/PhD_FraudDetection/data/defi/elliptic_merged.csv"
df = pd.read_csv(path)

print("Loaded dataset:", path)
print("Original shape:", df.shape)

# Identify label
label_col = "class"
y_original = df[label_col].values

print("\nOriginal label values:", np.unique(y_original))

# =====================================================
# STEP 1 — RE-ENCODE LABELS 1,2,3 → 0,1,2
# =====================================================
y = y_original - 1

print("Re-encoded label values:", np.unique(y))

# Drop non-feature columns
X = df.drop(columns=[label_col, "txId", "Time step"], errors="ignore")

# =====================================================
# STEP 2 — IMPUTE MISSING VALUES
# =====================================================
imp = SimpleImputer(strategy="mean")
X_imputed = imp.fit_transform(X)

# =====================================================
# STEP 3 — SCALING
# =====================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print("\nFinal X_scaled shape:", X_scaled.shape)
print("Final y shape:", y.shape)
print("Preprocessing complete — labels fixed.")


Loaded dataset: /content/drive/MyDrive/PhD_FraudDetection/data/defi/elliptic_merged.csv
Original shape: (203769, 185)

Original label values: [1 2 3]
Re-encoded label values: [0 1 2]

Final X_scaled shape: (203769, 182)
Final y shape: (203769,)
Preprocessing complete — labels fixed.


In [ ]:
# ================================================================
# FEDERATED LEARNING SETUP — DEFI (ELLIPTIC DATASET, FIXED LABELS)
# ================================================================

import numpy as np
import os
from sklearn.model_selection import StratifiedKFold, train_test_split

# Output directory
RESULTS_DIR = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model"
os.makedirs(RESULTS_DIR, exist_ok=True)

OUT_CSV = f"{RESULTS_DIR}/federated_seed_based_results.csv"

# Number of federated clients
NUM_CLIENTS = 5

# Seed list (same as centralized phase)
seed_list = [1011, 2022, 3033, 4044, 5055]

# ================================================================
# FIXED stratified client split — ensures ALL classes appear in ALL clients
# ================================================================
def stratified_client_split(X, y, n_clients=5, random_state=None):
    skf = StratifiedKFold(n_splits=n_clients, shuffle=True, random_state=random_state)
    clients = []
    X_np = np.asarray(X)
    y_np = np.asarray(y)

    for _, idx in skf.split(X_np, y_np):
        clients.append((X_np[idx], y_np[idx]))

    return clients

print("FL setup ready.")
print("NUM_CLIENTS =", NUM_CLIENTS)
print("Seeds =", seed_list)
print("Global classes =", np.unique(y))


FL setup ready.
NUM_CLIENTS = 5
Seeds = [1011, 2022, 3033, 4044, 5055]
Global classes = [0 1 2]


In [ ]:
# ================================================================
# FEDERATED LR (FedAvg) — RE-RUN (DEFI ELLIPTIC, SEED-BASED)
# ================================================================
from copy import deepcopy
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

# ensure results list starts fresh if OUT_CSV missing, else load existing to append
if os.path.exists(OUT_CSV):
    results = pd.read_csv(OUT_CSV).to_dict(orient="records")
else:
    results = []

print("\n=== Federated LR (FedAvg) — Seed-based (ELLIPTIC DATASET) ===")

for seed in seed_list:
    print(f"\n--- Seed {seed} ---")
    clients = stratified_client_split(X_scaled, y, n_clients=NUM_CLIENTS, random_state=seed)

    X_train_glob, X_test_glob, y_train_glob, y_test_glob = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    local_weights = []
    classes_ = None
    for i, (Xc, yc) in enumerate(clients):
        lm = LogisticRegression(max_iter=1200)
        lm.fit(Xc, yc)
        local_weights.append((lm.coef_.copy(), lm.intercept_.copy()))
        classes_ = lm.classes_

    # FedAvg
    coefs = np.array([w[0] for w in local_weights])
    inters = np.array([w[1] for w in local_weights])
    avg_coef = coefs.mean(axis=0)
    avg_inter = inters.mean(axis=0)

    global_lr = LogisticRegression(max_iter=1200)
    try:
        global_lr.fit(X_train_glob[:2], y_train_glob[:2])
    except:
        global_lr.classes_ = np.unique(y)

    global_lr.coef_ = avg_coef
    global_lr.intercept_ = avg_inter
    global_lr.classes_ = classes_ if classes_ is not None else np.unique(y)

    y_proba_all = global_lr.predict_proba(X_test_glob)
    y_proba = y_proba_all[:,1] if y_proba_all.shape[1] == 2 else y_proba_all
    y_pred = global_lr.predict(X_test_glob)

    m = compute_metrics(y_test_glob, y_pred, y_proba)
    m.update({"Model": "LR_FedAvg", "Seed": seed})
    results.append(m)

    print(f"[LR_FedAvg] Seed={seed} | Acc={m['Accuracy']:.4f} | Prec={m['Precision']:.4f} | F1={m['F1']:.4f} | AUC={m['AUC']:.4f}")

# save
pd.DataFrame(results).to_csv(OUT_CSV, index=False)
print("\nSaved LR FedAvg results to:", OUT_CSV)



=== Federated LR (FedAvg) — Seed-based (ELLIPTIC DATASET) ===

--- Seed 1011 ---
[LR_FedAvg] Seed=1011 | Acc=0.8534 | Prec=0.8321 | F1=0.8330 | AUC=0.8836

--- Seed 2022 ---
[LR_FedAvg] Seed=2022 | Acc=0.8549 | Prec=0.8390 | F1=0.8346 | AUC=0.8855

--- Seed 3033 ---
[LR_FedAvg] Seed=3033 | Acc=0.8572 | Prec=0.8392 | F1=0.8373 | AUC=0.8851

--- Seed 4044 ---
[LR_FedAvg] Seed=4044 | Acc=0.8503 | Prec=0.8339 | F1=0.8291 | AUC=0.8793

--- Seed 5055 ---
[LR_FedAvg] Seed=5055 | Acc=0.8560 | Prec=0.8417 | F1=0.8359 | AUC=0.8834

Saved LR FedAvg results to: /content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/federated_seed_based_results.csv


In [ ]:
# ================================================================
# FEDERATED RF (Majority Vote with alignment) — RE-RUN
# DEFI — ELLIPTIC DATASET — SEED-BASED
# ================================================================
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd

print("\n=== Federated RF (Majority Vote + Aligned Prob) — Seed-based (ELLIPTIC DATASET) ===")

# load existing results list
if os.path.exists(OUT_CSV):
    results = pd.read_csv(OUT_CSV).to_dict(orient="records")
else:
    results = []

global_classes = np.unique(y)  # [0,1,2]

def majority_vote_from_local_preds(local_preds_array, tie_breaker_probas=None):
    n_clients, n_samples = local_preds_array.shape
    final_preds = np.zeros(n_samples, dtype=int)
    for j in range(n_samples):
        votes = local_preds_array[:, j]
        counts = np.bincount(votes, minlength=len(global_classes))
        top = np.flatnonzero(counts == counts.max())
        if len(top) == 1:
            final_preds[j] = top[0]
        else:
            if tie_breaker_probas is not None:
                final_preds[j] = np.argmax(tie_breaker_probas[j])
            else:
                final_preds[j] = top[0]
    return final_preds

for seed in seed_list:
    print(f"\n--- Seed {seed} ---")
    clients = stratified_client_split(X_scaled, y, n_clients=NUM_CLIENTS, random_state=seed)
    X_train_glob, X_test_glob, y_train_glob, y_test_glob = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    local_preds = []
    aligned_probs = []
    for i, (Xc, yc) in enumerate(clients):
        rm = RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=seed + i)
        rm.fit(Xc, yc)
        pred_c = rm.predict(X_test_glob)
        local_preds.append(pred_c)

        proba = rm.predict_proba(X_test_glob)
        aligned = np.zeros((len(X_test_glob), len(global_classes)))
        for idx, cls in enumerate(rm.classes_):
            target_col = np.where(global_classes == cls)[0][0]
            aligned[:, target_col] = proba[:, idx]
        aligned_probs.append(aligned)

    local_preds_array = np.vstack(local_preds)
    avg_proba = np.mean(np.stack(aligned_probs, axis=0), axis=0)
    vote_preds = majority_vote_from_local_preds(local_preds_array, tie_breaker_probas=avg_proba)

    # fallback safety
    invalid_mask = (vote_preds >= len(global_classes))
    if invalid_mask.any():
        vote_preds[invalid_mask] = np.argmax(avg_proba[invalid_mask], axis=1)

    y_proba_for_auc = avg_proba if avg_proba.shape[1] > 2 else avg_proba[:,1]

    m = compute_metrics(y_test_glob, vote_preds, y_proba_for_auc)
    m.update({"Model": "RF_Fed_MajorityVote_FixedLabels", "Seed": seed})
    results.append(m)

    print(f"[RF_Fed_MajorityVote_FixedLabels] Seed={seed} | Acc={m['Accuracy']:.4f} | Prec={m['Precision']:.4f} | F1={m['F1']:.4f} | AUC={m['AUC']:.4f}")

# save
pd.DataFrame(results).to_csv(OUT_CSV, index=False)
print("\nSaved RF federated results to:", OUT_CSV)



=== Federated RF (Majority Vote + Aligned Prob) — Seed-based (ELLIPTIC DATASET) ===

--- Seed 1011 ---
[RF_Fed_MajorityVote_FixedLabels] Seed=1011 | Acc=0.9462 | Prec=0.9458 | F1=0.9447 | AUC=0.9951

--- Seed 2022 ---
[RF_Fed_MajorityVote_FixedLabels] Seed=2022 | Acc=0.9487 | Prec=0.9483 | F1=0.9472 | AUC=0.9954

--- Seed 3033 ---
[RF_Fed_MajorityVote_FixedLabels] Seed=3033 | Acc=0.9465 | Prec=0.9460 | F1=0.9449 | AUC=0.9951

--- Seed 4044 ---
[RF_Fed_MajorityVote_FixedLabels] Seed=4044 | Acc=0.9444 | Prec=0.9440 | F1=0.9427 | AUC=0.9950

--- Seed 5055 ---
[RF_Fed_MajorityVote_FixedLabels] Seed=5055 | Acc=0.9468 | Prec=0.9463 | F1=0.9452 | AUC=0.9953

Saved RF federated results to: /content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/federated_seed_based_results.csv


In [ ]:
# ================================================================
# FEDERATED XGBOOST (Aligned Probabilities + Average)
# DEFI — ELLIPTIC DATASET — SEED-BASED
# ================================================================
import xgboost as xgb
import numpy as np
import pandas as pd

print("\n=== Federated XGB (Prob-Avg with alignment) — Seed-based (ELLIPTIC DATASET) ===")

# load existing results list
if os.path.exists(OUT_CSV):
    results = pd.read_csv(OUT_CSV).to_dict(orient="records")
else:
    results = []

global_classes = np.unique(y)

for seed in seed_list:
    print(f"\n--- Seed {seed} ---")
    clients = stratified_client_split(X_scaled, y, n_clients=NUM_CLIENTS, random_state=seed)
    X_train_glob, X_test_glob, y_train_glob, y_test_glob = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    aligned_probs = []
    for i, (Xc, yc) in enumerate(clients):
        xm = xgb.XGBClassifier(n_estimators=150, verbosity=0, random_state=seed + i, eval_metric="mlogloss")
        xm.fit(Xc, yc)
        proba = xm.predict_proba(X_test_glob)
        aligned = np.zeros((len(X_test_glob), len(global_classes)))
        for idx, cls in enumerate(xm.classes_):
            target_col = np.where(global_classes == cls)[0][0]
            aligned[:, target_col] = proba[:, idx]
        aligned_probs.append(aligned)

    avg_proba = np.mean(np.stack(aligned_probs, axis=0), axis=0)
    if avg_proba.shape[1] > 2:
        y_pred = np.argmax(avg_proba, axis=1)
        y_proba_for_auc = avg_proba
    else:
        y_pred = (avg_proba[:,1] >= 0.5).astype(int)
        y_proba_for_auc = avg_proba[:,1]

    m = compute_metrics(y_test_glob, y_pred, y_proba_for_auc)
    m.update({"Model": "XGB_FedProb_FixedLabels", "Seed": seed})
    results.append(m)

    print(f"[XGB_FedProb_FixedLabels] Seed={seed} | Acc={m['Accuracy']:.4f} | Prec={m['Precision']:.4f} | F1={m['F1']:.4f} | AUC={m['AUC']:.4f}")

# save
pd.DataFrame(results).to_csv(OUT_CSV, index=False)
print("\nSaved XGB federated results to:", OUT_CSV)



=== Federated XGB (Prob-Avg with alignment) — Seed-based (ELLIPTIC DATASET) ===

--- Seed 1011 ---
[XGB_FedProb_FixedLabels] Seed=1011 | Acc=0.9624 | Prec=0.9620 | F1=0.9618 | AUC=0.9968

--- Seed 2022 ---
[XGB_FedProb_FixedLabels] Seed=2022 | Acc=0.9633 | Prec=0.9629 | F1=0.9627 | AUC=0.9971

--- Seed 3033 ---


KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

dir_path = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model"

print("Directory exists:", os.path.exists(dir_path))
print("\nFiles inside directory:")
print(os.listdir(dir_path))


Directory exists: True

Files inside directory:
['LR_seed_results.csv', 'RF_seed_results.csv', 'XGB_seed_results.csv', 'DeFi_Centralized_Seed_Results.pdf', 'random_run_centralized_results.csv', 'DeFi_Centralized_RandomRun_Results.pdf', 'federated_seed_based_results.csv']


In [ ]:
# ================================================================
# Generate PDF for DeFi Federated Learning (Seed-Based Results)
# ================================================================
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
import pandas as pd
import os

# Load CSV
csv_path = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/federated_seed_based_results.csv"
df = pd.read_csv(csv_path)

# Filter only FL seed-based models
models = ["LR_FedAvg", "RF_Fed_MajorityVote_FixedLabels", "XGB_FedProb_FixedLabels"]
df = df[df["Model"].isin(models)]
df = df.sort_values(["Model", "Seed"])

# PDF output path
pdf_path = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/DeFi_FL_Seed_Results.pdf"

styles = getSampleStyleSheet()
story = []

# Title
story.append(Paragraph("DeFi (Elliptic) – Federated Learning (Seed-Based Results)", styles['Title']))
story.append(Spacer(1, 12))

# Add tables for each model
for model in models:
    story.append(Paragraph(f"<b>{model}</b>", styles['Heading2']))

    sub = df[df["Model"] == model]
    table_data = [["Seed", "Accuracy", "Precision", "F1 Score", "AUC"]]

    for _, row in sub.iterrows():
        table_data.append([
            int(row["Seed"]),
            f"{row['Accuracy']:.4f}",
            f"{row['Precision']:.4f}",
            f"{row['F1']:.4f}",
            f"{row['AUC']:.4f}"
        ])

    table = Table(table_data, colWidths=[70, 70, 70, 70, 70])
    table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.lightgrey),
        ('TEXTCOLOR', (0,0), (-1,0), colors.black),
        ('ALIGN', (0,0), (-1,-1), 'CENTER'),
        ('FONT', (0,0), (-1,-1), 'Helvetica'),
        ('GRID', (0,0), (-1,-1), 0.5, colors.black),
    ]))

    story.append(table)
    story.append(Spacer(1, 18))

# Build PDF
doc = SimpleDocTemplate(pdf_path, pagesize=letter)
doc.build(story)

print("PDF generated successfully:")
print(pdf_path)


ModuleNotFoundError: No module named 'reportlab'

In [ ]:
!pip install reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 20.6 MB/s eta 0:00:00


In [ ]:
# ================================================================
# Generate PDF for DeFi Federated Learning (Seed-Based Results)
# ================================================================
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
import pandas as pd
import os

# Load CSV
csv_path = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/federated_seed_based_results.csv"
df = pd.read_csv(csv_path)

# Filter only FL seed-based models
models = ["LR_FedAvg", "RF_Fed_MajorityVote_FixedLabels", "XGB_FedProb_FixedLabels"]
df = df[df["Model"].isin(models)]
df = df.sort_values(["Model", "Seed"])

# PDF output path
pdf_path = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/DeFi_FL_Seed_Results.pdf"

styles = getSampleStyleSheet()
story = []

# Title
story.append(Paragraph("DeFi (Elliptic) – Federated Learning (Seed-Based Results)", styles['Title']))
story.append(Spacer(1, 12))

# Add tables for each model
for model in models:
    story.append(Paragraph(f"<b>{model}</b>", styles['Heading2']))

    sub = df[df["Model"] == model]
    table_data = [["Seed", "Accuracy", "Precision", "F1 Score", "AUC"]]

    for _, row in sub.iterrows():
        table_data.append([
            int(row["Seed"]),
            f"{row['Accuracy']:.4f}",
            f"{row['Precision']:.4f}",
            f"{row['F1']:.4f}",
            f"{row['AUC']:.4f}"
        ])

    table = Table(table_data, colWidths=[70, 70, 70, 70, 70])
    table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.lightgrey),
        ('TEXTCOLOR', (0,0), (-1,0), colors.black),
        ('ALIGN', (0,0), (-1,-1), 'CENTER'),
        ('FONT', (0,0), (-1,-1), 'Helvetica'),
        ('GRID', (0,0), (-1,-1), 0.5, colors.black),
    ]))

    story.append(table)
    story.append(Spacer(1, 18))

# Build PDF
doc = SimpleDocTemplate(pdf_path, pagesize=letter)
doc.build(story)

print("PDF generated successfully:")
print(pdf_path)


PDF generated successfully:
/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/DeFi_FL_Seed_Results.pdf


In [ ]:
# ================================================================
# DEFI (ELLIPTIC) — FEDERATED LEARNING RANDOM RUNS (5 AUTO SEEDS)
# LR FedAvg + RF Majority Vote + XGB ProbAvg
# ================================================================
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

import os

SAVE_PATH = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/federated_random_run_results.csv"

# Generate 5 auto random seeds
rng = np.random.default_rng(42)
random_seeds = rng.integers(10000, 999999, size=5).tolist()

print("=== AUTO-GENERATED RANDOM SEEDS FOR FL RANDOM RUN ===")
print(random_seeds, "\n")

results = []

global_classes = np.unique(y)

# ============================
# 1. LR FEDAVG
# ============================
print("===== FL RANDOM RUN — LR FedAvg =====")

for seed in random_seeds:
    print(f"\n--- Random Seed {seed} ---")

    clients = stratified_client_split(X_scaled, y, n_clients=NUM_CLIENTS, random_state=seed)

    # global test set
    X_train_glob, X_test_glob, y_train_glob, y_test_glob = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    local_weights = []
    classes_ = None
    for i, (Xc, yc) in enumerate(clients):
        lr = LogisticRegression(max_iter=1200)
        lr.fit(Xc, yc)
        local_weights.append((lr.coef_.copy(), lr.intercept_.copy()))
        classes_ = lr.classes_

    # FedAvg
    coefs = np.array([w[0] for w in local_weights])
    inters = np.array([w[1] for w in local_weights])
    avg_coef = coefs.mean(axis=0)
    avg_inter = inters.mean(axis=0)

    global_lr = LogisticRegression(max_iter=1200)
    try:
        global_lr.fit(X_train_glob[:2], y_train_glob[:2])
    except:
        global_lr.classes_ = np.unique(y)

    global_lr.coef_ = avg_coef
    global_lr.intercept_ = avg_inter
    global_lr.classes_ = classes_

    y_proba = global_lr.predict_proba(X_test_glob)
    y_pred = global_lr.predict(X_test_glob)

    m = compute_metrics(y_test_glob, y_pred, y_proba)
    m.update({"Random_Seed": seed, "Model": "LR_FedAvg"})
    results.append(m)

    print(f"[LR_FedAvg] Seed={seed} | Acc={m['Accuracy']:.4f} | Prec={m['Precision']:.4f} | F1={m['F1']:.4f} | AUC={m['AUC']:.4f}")


# ============================
# 2. RF FEDERATED (Majority Vote + aligned probs)
# ============================
print("\n===== FL RANDOM RUN — RF Federated (Majority Vote) =====")

def majority_vote(votes_matrix, tie_breaker_proba=None):
    n_clients, n_samples = votes_matrix.shape
    final = np.zeros(n_samples, dtype=int)

    for j in range(n_samples):
        votes = votes_matrix[:, j]
        counts = np.bincount(votes, minlength=len(global_classes))
        top = np.flatnonzero(counts == counts.max())

        if len(top) == 1:
            final[j] = top[0]
        else:
            if tie_breaker_proba is not None:
                final[j] = np.argmax(tie_breaker_proba[j])
            else:
                final[j] = top[0]

    return final

for seed in random_seeds:
    print(f"\n--- Random Seed {seed} ---")

    clients = stratified_client_split(X_scaled, y, n_clients=NUM_CLIENTS, random_state=seed)
    X_train_glob, X_test_glob, y_train_glob, y_test_glob = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    local_preds = []
    aligned_probs = []

    for i, (Xc, yc) in enumerate(clients):
        rf = RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=seed+i)
        rf.fit(Xc, yc)

        pred_client = rf.predict(X_test_glob)
        local_preds.append(pred_client)

        proba = rf.predict_proba(X_test_glob)
        aligned = np.zeros((len(X_test_glob), len(global_classes)))

        for idx, cls in enumerate(rf.classes_):
            target_col = np.where(global_classes == cls)[0][0]
            aligned[:, target_col] = proba[:, idx]

        aligned_probs.append(aligned)

    votes_matrix = np.vstack(local_preds)
    avg_proba = np.mean(np.stack(aligned_probs, axis=0), axis=0)

    final_pred = majority_vote(votes_matrix, tie_breaker_proba=avg_proba)

    m = compute_metrics(y_test_glob, final_pred, avg_proba)
    m.update({"Random_Seed": seed, "Model": "RF_Fed_MajorityVote"})
    results.append(m)

    print(f"[RF_Fed] Seed={seed} | Acc={m['Accuracy']:.4f} | Prec={m['Precision']:.4f} | F1={m['F1']:.4f} | AUC={m['AUC']:.4f}")


# ============================
# 3. XGB FEDERATED (probability avg)
# ============================
print("\n===== FL RANDOM RUN — XGB Federated (ProbAvg) =====")

for seed in random_seeds:
    print(f"\n--- Random Seed {seed} ---")

    clients = stratified_client_split(X_scaled, y, n_clients=NUM_CLIENTS, random_state=seed)
    X_train_glob, X_test_glob, y_train_glob, y_test_glob = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=seed
    )

    aligned_probs = []

    for i, (Xc, yc) in enumerate(clients):
        xm = xgb.XGBClassifier(n_estimators=150, verbosity=0, random_state=seed+i, eval_metric="mlogloss")
        xm.fit(Xc, yc)

        proba = xm.predict_proba(X_test_glob)
        aligned = np.zeros((len(X_test_glob), len(global_classes)))

        for idx, cls in enumerate(xm.classes_):
            target_col = np.where(global_classes == cls)[0][0]
            aligned[:, target_col] = proba[:, idx]

        aligned_probs.append(aligned)

    avg_proba = np.mean(np.stack(aligned_probs, axis=0), axis=0)
    y_pred = np.argmax(avg_proba, axis=1)

    m = compute_metrics(y_test_glob, y_pred, avg_proba)
    m.update({"Random_Seed": seed, "Model": "XGB_FedProbAvg"})
    results.append(m)

    print(f"[XGB_Fed] Seed={seed} | Acc={m['Accuracy']:.4f} | Prec={m['Precision']:.4f} | F1={m['F1']:.4f} | AUC={m['AUC']:.4f}")


# ============================
# SAVE RESULTS
# ============================
pd.DataFrame(results).to_csv(SAVE_PATH, index=False)
print("\nRandom-Run Federated Results Saved To:")
print(SAVE_PATH)


=== AUTO-GENERATED RANDOM SEEDS FOR FL RANDOM RUN ===
[98358, 776215, 658025, 444489, 438684] 

===== FL RANDOM RUN — LR FedAvg =====

--- Random Seed 98358 ---
[LR_FedAvg] Seed=98358 | Acc=0.8560 | Prec=0.8373 | F1=0.8359 | AUC=0.8852

--- Random Seed 776215 ---
[LR_FedAvg] Seed=776215 | Acc=0.8556 | Prec=0.8328 | F1=0.8354 | AUC=0.8867

--- Random Seed 658025 ---
[LR_FedAvg] Seed=658025 | Acc=0.8543 | Prec=0.8365 | F1=0.8341 | AUC=0.8824

--- Random Seed 444489 ---
[LR_FedAvg] Seed=444489 | Acc=0.8541 | Prec=0.8340 | F1=0.8338 | AUC=0.8852

--- Random Seed 438684 ---
[LR_FedAvg] Seed=438684 | Acc=0.8554 | Prec=0.8348 | F1=0.8353 | AUC=0.8868

===== FL RANDOM RUN — RF Federated (Majority Vote) =====

--- Random Seed 98358 ---
[RF_Fed] Seed=98358 | Acc=0.9462 | Prec=0.9457 | F1=0.9447 | AUC=0.9952

--- Random Seed 776215 ---
[RF_Fed] Seed=776215 | Acc=0.9464 | Prec=0.9460 | F1=0.9449 | AUC=0.9951

--- Random Seed 658025 ---
[RF_Fed] Seed=658025 | Acc=0.9443 | Prec=0.9437 | F1=0.9427 | 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

path = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/"
os.listdir(path)


['LR_seed_results.csv',
 'RF_seed_results.csv',
 'XGB_seed_results.csv',
 'DeFi_Centralized_Seed_Results.pdf',
 'random_run_centralized_results.csv',
 'DeFi_Centralized_RandomRun_Results.pdf',
 'federated_seed_based_results.csv',
 'DeFi_FL_Seed_Results.pdf',
 'federated_random_run_results.csv']

In [ ]:
# ================================================================
# PDF: DeFi Federated Learning - Random Run Results (Elliptic Dataset)
# ================================================================
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
import pandas as pd

# Correct file paths
csv_path = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/federated_random_run_results.csv"
pdf_path = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/DeFi_FL_RandomRun_Results.pdf"

# Load the CSV
df = pd.read_csv(csv_path)

# Create PDF content
styles = getSampleStyleSheet()
story = []

title = Paragraph("<b>DeFi Federated Learning - Random Run Results (Elliptic Dataset)</b>", styles["Title"])
story.append(title)
story.append(Spacer(1, 20))

# Table structure
table_data = [["Random Seed", "Model", "Accuracy", "Precision", "F1-Score", "AUC"]]

# Add rows from CSV
for _, row in df.iterrows():
    table_data.append([
        str(row["Random_Seed"]),
        row["Model"],
        f"{row['Accuracy']:.4f}",
        f"{row['Precision']:.4f}",
        f"{row['F1']:.4f}",
        f"{row['AUC']:.4f}",
    ])

# Create table
table = Table(table_data, repeatRows=1)
table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
    ("ALIGN", (0, 0), (-1, -1), "CENTER"),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("FONTSIZE", (0, 0), (-1, 0), 11),
    ("BOTTOMPADDING", (0, 0), (-1, 0), 8),
]))

story.append(table)

# Generate PDF
doc = SimpleDocTemplate(pdf_path, pagesize=letter)
doc.build(story)

pdf_path


'/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/DeFi_FL_RandomRun_Results.pdf'

In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/federated_random_run_results.csv")
df.head()


,Accuracy,Precision,F1,AUC,Random_Seed,Model
0,0.856014,0.837274,0.835921,0.885215,98358,LR_FedAvg
1,0.855572,0.832756,0.835404,0.886681,776215,LR_FedAvg
2,0.854272,0.836531,0.834052,0.882415,658025,LR_FedAvg
3,0.854125,0.834009,0.833776,0.885191,444489,LR_FedAvg
4,0.855401,0.834810,0.835349,0.886756,438684,LR_FedAvg


In [ ]:
# ================================================================
# PDF: DeFi Federated Learning - Random Run Results (Three Tables)
# ================================================================
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
import pandas as pd

# Paths
csv_path = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/federated_random_run_results.csv"
pdf_path = "/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/DeFi_FL_RandomRun_Results.pdf"

# Load dataset
df = pd.read_csv(csv_path)

styles = getSampleStyleSheet()
story = []

# ============================
# Title
# ============================
title = Paragraph("<b>DeFi Federated Learning - Random Run Results (Elliptic Dataset)</b>", styles["Title"])
story.append(title)
story.append(Spacer(1, 20))

# ============================
# FUNCTION TO ADD A TABLE
# ============================
def add_model_table(model_name, display_name):
    story.append(Paragraph(f"<b>{display_name}</b>", styles["Heading2"]))
    story.append(Spacer(1, 10))

    subset = df[df["Model"] == model_name]

    table_data = [["Random Seed", "Accuracy", "Precision", "F1-Score", "AUC"]]

    for _, row in subset.iterrows():
        table_data.append([
            str(row["Random_Seed"]),
            f"{row['Accuracy']:.4f}",
            f"{row['Precision']:.4f}",
            f"{row['F1']:.4f}",
            f"{row['AUC']:.4f}",
        ])

    table = Table(table_data, repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("BOTTOMPADDING", (0, 0), (-1, 0), 8),
    ]))

    story.append(table)
    story.append(Spacer(1, 20))


# ============================
# Add three model-specific tables
# ============================
add_model_table("LR_FedAvg", "Logistic Regression (FedAvg)")
add_model_table("RF_Fed_MajorityVote", "Random Forest (Federated Majority Vote)")
add_model_table("XGB_FedProbAvg", "XGBoost (Federated Probability Averaging)")

# ============================
# Create the PDF
# ============================
doc = SimpleDocTemplate(pdf_path, pagesize=letter)
doc.build(story)

pdf_path


'/content/drive/MyDrive/PhD_FraudDetection/results_defi_centralized_by_model/DeFi_FL_RandomRun_Results.pdf'